In [1]:
"""
PM2.5 Model Performance Visualisations
=======================================
Loads all saved pkl models and produces publication-quality plots.

Plots saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\

Run:
    pip install matplotlib seaborn scipy
    python pm25_plots.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from scipy import stats
from sklearn.metrics import (r2_score, mean_absolute_error,
                             mean_squared_error, mean_absolute_percentage_error)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\src\models")
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.80

# ── Palette (colorblind-safe) ──────────────────────────────────────────────
MODEL_COLORS = {
    "XGBoost"         : "#E63946",
    "LightGBM"        : "#F4A261",
    "RandomForest"    : "#2A9D8F",
    "GradientBoosting": "#457B9D",
    "CatBoost"        : "#6A0572",
    "Ensemble"        : "#1D3557",
}
SPLIT_COLORS = {"train": "#2A9D8F", "val": "#F4A261", "test": "#E63946"}

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F8F9FA",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.2,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
})

def save(fig, name):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p.name}")

# ════════════════════════════════════════════
# 1. DATA + FEATURES  (identical to pipeline)
# ════════════════════════════════════════════
print("=" * 60)
print("  PM2.5 Visualisation Suite")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_name", "date"]).reset_index(drop=True)

grp = df.groupby("station_name")["pm25"]
for lag in [1, 2, 3, 7, 14]:
    df[f"pm25_lag{lag}"] = grp.shift(lag)
for window in [3, 7, 14, 30]:
    df[f"pm25_roll{window}"] = (
        grp.shift(1).transform(lambda x: x.rolling(window, min_periods=1).mean()))
df["pm25_ewm7"]            = grp.shift(1).transform(lambda x: x.ewm(span=7,  min_periods=1).mean())
df["pm25_ewm14"]           = grp.shift(1).transform(lambda x: x.ewm(span=14, min_periods=1).mean())
df["pm25_lag1_diff"]       = df["pm25_lag1"]  - df["pm25_lag2"]
df["pm25_lag7_diff"]       = df["pm25_lag1"]  - df["pm25_lag7"]
df["pm25_roll3_diff"]      = df["pm25_roll3"] - df["pm25_roll7"]
df["pm25_lag1_vs_roll7"]   = df["pm25_lag1"]  - df["pm25_roll7"]
df["pm25_roll3_vs_roll14"] = df["pm25_roll3"] - df["pm25_roll14"]

df["station_enc"] = LabelEncoder().fit_transform(df["station_name"])
if "season" in df.columns:
    df["season_enc"] = LabelEncoder().fit_transform(df["season"].astype(str))

AOD_SENTINEL = ["AOD_mean","AOD_max","AOD_p75","AOD_lag1","AOD_lag2","AOD_roll3","AOD_roll7"]
for col in AOD_SENTINEL:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

DROP = {"pm25","date","station_name","season","lat","lon"}
FEATURE_COLS = [c for c in df.columns if c not in DROP and df[c].dtype != object]
TARGET = "pm25"
USE_RESID = "station_month_pm" in df.columns
if USE_RESID:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]

df = df.sort_values("date").reset_index(drop=True)
n  = len(df)
t1 = int(n * TRAIN_FRAC)
t2 = int(n * VAL_FRAC)

X_tr = df[FEATURE_COLS].iloc[:t1]
X_va = df[FEATURE_COLS].iloc[t1:t2]
X_te = df[FEATURE_COLS].iloc[t2:]
y_raw_tr = df[TARGET].iloc[:t1].values
y_raw_va = df[TARGET].iloc[t1:t2].values
y_raw_te = df[TARGET].iloc[t2:].values

if USE_RESID:
    y_tr = df["pm25_resid"].iloc[:t1]
    y_va = df["pm25_resid"].iloc[t1:t2]
    base_tr = df["station_month_pm"].iloc[:t1].values
    base_va = df["station_month_pm"].iloc[t1:t2].values
    base_te = df["station_month_pm"].iloc[t2:].values

imp_path = MODEL_DIR / "imputer.pkl"
_raw = joblib.load(imp_path) if imp_path.exists() else None
if _raw and hasattr(_raw, "transform") and hasattr(_raw, "statistics_"):
    imp = _raw
else:
    imp = SimpleImputer(strategy="median").fit(X_tr)

X_tr_i = pd.DataFrame(imp.transform(X_tr), columns=FEATURE_COLS)
X_va_i = pd.DataFrame(imp.transform(X_va), columns=FEATURE_COLS)
X_te_i = pd.DataFrame(imp.transform(X_te), columns=FEATURE_COLS)

# ── Load models ────────────────────────────────────────────────────────────
PKL = {"XGBoost":"xgboost.pkl","LightGBM":"lightgbm.pkl",
       "RandomForest":"randomforest.pkl","GradientBoosting":"gradientboosting.pkl",
       "CatBoost":"catboost.pkl"}

models = {}
for name, fname in PKL.items():
    p = MODEL_DIR / fname
    if p.exists():
        models[name] = joblib.load(p)
        print(f"  Loaded : {name}")

def predict(model, X, base=None):
    p = model.predict(X)
    if USE_RESID and base is not None:
        p = p + base
    return np.maximum(p, 0)

def metrics(yt, yp):
    return dict(R2   = r2_score(yt, yp),
                MAE  = mean_absolute_error(yt, yp),
                RMSE = mean_squared_error(yt, yp) ** 0.5,
                MAPE = mean_absolute_percentage_error(yt, yp) * 100)

# ── Collect all predictions & metrics ──────────────────────────────────────
all_preds  = {}
all_metrics = {}

for name, model in models.items():
    tr_pred = predict(model, X_tr_i, base_tr if USE_RESID else None)
    va_pred = predict(model, X_va_i, base_va if USE_RESID else None)
    te_pred = predict(model, X_te_i, base_te if USE_RESID else None)
    all_preds[name]   = {"train": tr_pred, "val": va_pred, "test": te_pred}
    all_metrics[name] = {"train": metrics(y_raw_tr, tr_pred),
                         "val"  : metrics(y_raw_va, va_pred),
                         "test" : metrics(y_raw_te, te_pred)}

# ── Blend ensemble ─────────────────────────────────────────────────────────
blend_path = MODEL_DIR / "blend_meta.pkl"
if blend_path.exists():
    meta = joblib.load(blend_path)
    mnames = meta["model_names"]
    bw     = meta["blend_weights"]
    ens_te = np.maximum(
        sum(bw[i] * all_preds[mnames[i]]["test"] for i in range(len(mnames))
            if mnames[i] in all_preds), 0)
    all_preds["Ensemble"]   = {"test": ens_te}
    all_metrics["Ensemble"] = {"test": metrics(y_raw_te, ens_te)}

dates_te = df["date"].iloc[t2:].values
print(f"\n  Generating plots …")

# ════════════════════════════════════════════
# PLOT 1 — Metrics Comparison Bar Chart (R², MAE, RMSE per model × split)
# ════════════════════════════════════════════
splits = ["train", "val", "test"]
metric_names = ["R2", "MAE", "RMSE"]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Model Performance Across Train / Val / Test", fontsize=15, fontweight="bold", y=1.02)

for ax, met in zip(axes, metric_names):
    x      = np.arange(len(models))
    width  = 0.22
    for j, split in enumerate(splits):
        vals = [all_metrics[n][split][met] for n in models]
        bars = ax.bar(x + (j - 1) * width, vals, width,
                      label=split.capitalize(),
                      color=SPLIT_COLORS[split], alpha=0.88,
                      edgecolor="white", linewidth=0.8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005 * max(vals),
                    f"{v:.2f}", ha="center", va="bottom",
                    fontsize=7.5, fontweight="bold")
    ax.set_title(met, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(list(models.keys()), rotation=25, ha="right")
    ax.legend(fontsize=8)
    if met == "R2":
        ax.set_ylim(0.7, 1.02)
    ax.set_ylabel(met)

plt.tight_layout()
save(fig, "01_metrics_comparison.png")

# ════════════════════════════════════════════
# PLOT 2 — Scatter: Predicted vs Actual (test set, per model)
# ════════════════════════════════════════════
n_models = len(models)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()
fig.suptitle("Predicted vs Actual PM2.5 — Test Set", fontsize=15, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    yp = all_preds[name]["test"]
    yt = y_raw_te
    color = MODEL_COLORS.get(name, "#555")

    ax.scatter(yt, yp, alpha=0.25, s=8, color=color, rasterized=True)

    # Perfect prediction line
    lim = max(yt.max(), yp.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1.2, label="Perfect fit")

    # Regression line
    slope, intercept, r, *_ = stats.linregress(yt, yp)
    xs = np.linspace(0, lim, 200)
    ax.plot(xs, slope * xs + intercept, color=color, lw=2,
            label=f"Fit  y={slope:.2f}x+{intercept:.1f}")

    m = all_metrics[name]["test"]
    ax.set_title(f"{name}\nR²={m['R2']:.4f}  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Actual PM2.5 (µg/m³)")
    ax.set_ylabel("Predicted PM2.5 (µg/m³)")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "02_scatter_predicted_vs_actual.png")

# ════════════════════════════════════════════
# PLOT 3 — Residual Distribution (test set)
# ════════════════════════════════════════════
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()
fig.suptitle("Residual Distribution — Test Set  (Actual − Predicted)", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    resid = y_raw_te - all_preds[name]["test"]
    color = MODEL_COLORS.get(name, "#555")
    ax.hist(resid, bins=60, color=color, alpha=0.75, edgecolor="white", linewidth=0.5)
    ax.axvline(0,       color="black", lw=1.5, ls="--", label="Zero error")
    ax.axvline(resid.mean(), color="red", lw=1.5, ls="-",
               label=f"Mean={resid.mean():.2f}")
    ax.set_title(f"{name}  (σ={resid.std():.2f})", fontweight="bold")
    ax.set_xlabel("Residual (µg/m³)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "03_residual_distribution.png")

# ════════════════════════════════════════════
# PLOT 4 — Overfit Gap Heatmap (R² across splits)
# ════════════════════════════════════════════
splits_order = ["train", "val", "test"]
r2_matrix = pd.DataFrame(
    {split: [all_metrics[n][split]["R2"] for n in models]
     for split in splits_order},
    index=list(models.keys())
)
gap_col = r2_matrix["train"] - r2_matrix["test"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={"width_ratios": [3, 1]})
fig.suptitle("Overfitting Analysis — R² Heatmap & Gap", fontsize=14, fontweight="bold")

# Heatmap
sns.heatmap(r2_matrix, ax=axes[0], annot=True, fmt=".4f",
            cmap="RdYlGn", vmin=0.75, vmax=1.0,
            linewidths=1, linecolor="white",
            cbar_kws={"label": "R²"})
axes[0].set_title("R² per Model × Split", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("")

# Gap bar
colors = ["#E63946" if g >= 0.15 else "#F4A261" if g >= 0.10 else "#2A9D8F"
          for g in gap_col]
bars = axes[1].barh(list(models.keys()), gap_col.values,
                    color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, gap_col):
    axes[1].text(v + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{v:.4f}", va="center", fontsize=9, fontweight="bold")
axes[1].axvline(0.10, color="#F4A261", lw=1.5, ls="--", label="Warn (0.10)")
axes[1].axvline(0.20, color="#E63946", lw=1.5, ls="--", label="Overfit (0.20)")
axes[1].set_title("Gap  (Train R² − Test R²)", fontweight="bold")
axes[1].set_xlabel("Gap")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 0.25)

plt.tight_layout()
save(fig, "04_overfit_gap_heatmap.png")

# ════════════════════════════════════════════
# PLOT 5 — Time Series: Actual vs All Models (test period, first 500 pts)
# ════════════════════════════════════════════
N_SHOW = 500
fig, ax = plt.subplots(figsize=(18, 6))
fig.suptitle(f"Time Series — Actual vs Predicted (Test set, first {N_SHOW} samples)",
             fontsize=14, fontweight="bold")

ax.plot(range(N_SHOW), y_raw_te[:N_SHOW], color="black",
        lw=1.8, label="Actual", zorder=10)

for name in models:
    color = MODEL_COLORS.get(name, "#aaa")
    ax.plot(range(N_SHOW), all_preds[name]["test"][:N_SHOW],
            color=color, lw=1.0, alpha=0.75, label=name)

if "Ensemble" in all_preds:
    ax.plot(range(N_SHOW), all_preds["Ensemble"]["test"][:N_SHOW],
            color=MODEL_COLORS["Ensemble"], lw=2.0, ls="--",
            alpha=0.9, label="Ensemble")

ax.set_xlabel("Sample index (test set)")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(fontsize=9, ncol=4, loc="upper right")
plt.tight_layout()
save(fig, "05_timeseries_actual_vs_predicted.png")

# ════════════════════════════════════════════
# PLOT 6 — Feature Importance (XGBoost, Top 25)
# ════════════════════════════════════════════
if "XGBoost" in models:
    xgb_m = models["XGBoost"]
    fi = pd.Series(xgb_m.feature_importances_, index=FEATURE_COLS).nlargest(25)

    fig, ax = plt.subplots(figsize=(10, 9))
    colors_fi = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(fi)))[::-1]
    bars = ax.barh(fi.index[::-1], fi.values[::-1],
                   color=colors_fi, edgecolor="white", linewidth=0.6)
    for bar, v in zip(bars, fi.values[::-1]):
        ax.text(v + 0.0005, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=8)
    ax.set_title("XGBoost — Top 25 Feature Importances (Test-set refit)",
                 fontweight="bold", fontsize=13)
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    save(fig, "06_feature_importance_xgboost.png")

# ════════════════════════════════════════════
# PLOT 7 — Error by PM2.5 Concentration Bin (test)
# ════════════════════════════════════════════
bins   = [0, 30, 60, 100, 150, 200, 300, 600]
labels = ["0–30", "30–60", "60–100", "100–150", "150–200", "200–300", "300+"]
bin_col = pd.cut(y_raw_te, bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Error by PM2.5 Concentration Bin — Test Set", fontsize=14, fontweight="bold")

# MAE per bin per model
mae_bin = {}
for name in models:
    resid = np.abs(y_raw_te - all_preds[name]["test"])
    mae_bin[name] = pd.Series(resid).groupby(bin_col).mean().values

x = np.arange(len(labels))
width = 0.15
for j, (name, vals) in enumerate(mae_bin.items()):
    offset = (j - len(models) / 2) * width + width / 2
    axes[0].bar(x + offset, vals, width, label=name,
                color=MODEL_COLORS.get(name, "#888"), alpha=0.85,
                edgecolor="white", linewidth=0.6)

axes[0].set_title("MAE by Concentration Bin", fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20)
axes[0].set_xlabel("PM2.5 Concentration Range (µg/m³)")
axes[0].set_ylabel("MAE (µg/m³)")
axes[0].legend(fontsize=8)

# Sample count per bin (context)
counts = pd.Series(y_raw_te).groupby(bin_col).count()
axes[1].bar(labels, counts.values, color="#457B9D", alpha=0.85,
            edgecolor="white", linewidth=0.8)
for i, (label, c) in enumerate(zip(labels, counts.values)):
    axes[1].text(i, c + 20, str(c), ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("Sample Count per Bin", fontweight="bold")
axes[1].set_xlabel("PM2.5 Concentration Range (µg/m³)")
axes[1].set_ylabel("Number of Samples")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
save(fig, "07_error_by_concentration_bin.png")

# ════════════════════════════════════════════
# PLOT 8 — Radar Chart: All Metrics Normalised
# ════════════════════════════════════════════
from matplotlib.patches import FancyArrowPatch

metric_labels = ["R²", "1−nMAE", "1−nRMSE", "1−nMAPE"]
# normalise to [0,1]: higher = better
te_r2   = np.array([all_metrics[n]["test"]["R2"]   for n in models])
te_mae  = np.array([all_metrics[n]["test"]["MAE"]  for n in models])
te_rmse = np.array([all_metrics[n]["test"]["RMSE"] for n in models])
te_mape = np.array([all_metrics[n]["test"]["MAPE"] for n in models])

def norm_inv(arr):    # lower is better → invert to [0,1]
    mn, mx = arr.min(), arr.max()
    return 1 - (arr - mn) / (mx - mn + 1e-9)

scores = np.column_stack([
    (te_r2  - te_r2.min())  / (te_r2.max()  - te_r2.min()  + 1e-9),
    norm_inv(te_mae),
    norm_inv(te_rmse),
    norm_inv(te_mape),
])

angles = np.linspace(0, 2 * np.pi, len(metric_labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
fig.suptitle("Model Comparison — Normalised Metrics (Test Set)",
             fontsize=14, fontweight="bold", y=1.01)

for i, name in enumerate(models):
    vals = scores[i].tolist() + scores[i][:1].tolist()
    ax.plot(angles, vals, lw=2, color=MODEL_COLORS.get(name, "#888"), label=name)
    ax.fill(angles, vals, alpha=0.07, color=MODEL_COLORS.get(name, "#888"))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=8)
ax.legend(loc="lower right", bbox_to_anchor=(1.35, -0.05), fontsize=9)
ax.grid(color="white", linewidth=1.2)
ax.set_facecolor("#F0F4F8")

plt.tight_layout()
save(fig, "08_radar_chart_metrics.png")

# ════════════════════════════════════════════
# PLOT 9 — Residual vs Predicted (heteroscedasticity check)
# ════════════════════════════════════════════
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()
fig.suptitle("Residuals vs Predicted — Heteroscedasticity Check (Test Set)",
             fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    yp    = all_preds[name]["test"]
    resid = y_raw_te - yp
    color = MODEL_COLORS.get(name, "#555")
    ax.scatter(yp, resid, alpha=0.2, s=7, color=color, rasterized=True)
    ax.axhline(0, color="black", lw=1.5, ls="--")
    # Smoothed trend
    sort_idx = np.argsort(yp)
    yp_s, resid_s = yp[sort_idx], resid[sort_idx]
    window = max(len(yp_s) // 30, 10)
    smooth = pd.Series(resid_s).rolling(window, center=True, min_periods=1).mean()
    ax.plot(yp_s, smooth, color="red", lw=2, label="Smoothed trend")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted PM2.5 (µg/m³)")
    ax.set_ylabel("Residual (µg/m³)")
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "09_residuals_vs_predicted.png")

# ════════════════════════════════════════════
# PLOT 10 — Summary Dashboard (single-page overview)
# ════════════════════════════════════════════
fig = plt.figure(figsize=(20, 14))
fig.suptitle("PM2.5 Model Performance Dashboard", fontsize=18,
             fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# [0,0] R² bar — test only
ax0 = fig.add_subplot(gs[0, 0])
names_list = list(models.keys())
r2_vals = [all_metrics[n]["test"]["R2"] for n in names_list]
colors  = [MODEL_COLORS.get(n, "#888") for n in names_list]
bars = ax0.bar(names_list, r2_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, r2_vals):
    ax0.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
             f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax0.set_ylim(0.75, 0.85)
ax0.set_title("Test R²", fontweight="bold")
ax0.set_ylabel("R²")
ax0.tick_params(axis="x", rotation=20)

# [0,1] MAE bar — test only
ax1 = fig.add_subplot(gs[0, 1])
mae_vals = [all_metrics[n]["test"]["MAE"] for n in names_list]
bars = ax1.bar(names_list, mae_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, mae_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.05,
             f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.set_title("Test MAE (µg/m³)", fontweight="bold")
ax1.set_ylabel("MAE")
ax1.tick_params(axis="x", rotation=20)

# [0,2] RMSE bar — test only
ax2 = fig.add_subplot(gs[0, 2])
rmse_vals = [all_metrics[n]["test"]["RMSE"] for n in names_list]
bars = ax2.bar(names_list, rmse_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, rmse_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.05,
             f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_title("Test RMSE (µg/m³)", fontweight="bold")
ax2.set_ylabel("RMSE")
ax2.tick_params(axis="x", rotation=20)

# [1,0:2] Time series — best model (CatBoost) + ensemble
ax3 = fig.add_subplot(gs[1, 0:2])
N = 300
ax3.plot(range(N), y_raw_te[:N], color="black", lw=2, label="Actual", zorder=5)
best_name = min(all_metrics, key=lambda n: all_metrics[n]["test"]["MAE"]
                if "test" in all_metrics[n] else 999)
ax3.plot(range(N), all_preds[best_name]["test"][:N],
         color=MODEL_COLORS.get(best_name, "#E63946"),
         lw=1.3, alpha=0.85, label=f"Best model ({best_name})")
if "Ensemble" in all_preds:
    ax3.plot(range(N), all_preds["Ensemble"]["test"][:N],
             color=MODEL_COLORS["Ensemble"], lw=1.5, ls="--",
             alpha=0.9, label="Ensemble")
ax3.set_title(f"Time Series — Actual vs Best Model + Ensemble (first {N} test samples)",
              fontweight="bold")
ax3.set_xlabel("Sample index")
ax3.set_ylabel("PM2.5 (µg/m³)")
ax3.legend(fontsize=9, loc="upper right")

# [1,2] Gap bar
ax4 = fig.add_subplot(gs[1, 2])
gaps  = [all_metrics[n]["train"]["R2"] - all_metrics[n]["test"]["R2"] for n in names_list]
gcols = ["#E63946" if g >= 0.15 else "#F4A261" if g >= 0.10 else "#2A9D8F" for g in gaps]
bars  = ax4.bar(names_list, gaps, color=gcols, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, gaps):
    ax4.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
             f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax4.axhline(0.10, color="#F4A261", lw=1.5, ls="--", label="Warn 0.10")
ax4.axhline(0.20, color="#E63946", lw=1.5, ls="--", label="Overfit 0.20")
ax4.set_title("Overfit Gap  (Train R² − Test R²)", fontweight="bold")
ax4.set_ylabel("Gap")
ax4.legend(fontsize=8)
ax4.tick_params(axis="x", rotation=20)

plt.tight_layout()
save(fig, "10_dashboard_summary.png")

# ════════════════════════════════════════════
# DONE
# ════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  All 10 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 60)
print("""
  01_metrics_comparison.png       — R², MAE, RMSE across splits
  02_scatter_predicted_vs_actual.png — Predicted vs Actual scatter
  03_residual_distribution.png    — Residual histograms
  04_overfit_gap_heatmap.png      — Heatmap + gap bar
  05_timeseries_actual_vs_predicted.png — Time series overlay
  06_feature_importance_xgboost.png — Top-25 XGBoost features
  07_error_by_concentration_bin.png — MAE per PM2.5 range
  08_radar_chart_metrics.png      — Normalised radar chart
  09_residuals_vs_predicted.png   — Heteroscedasticity check
  10_dashboard_summary.png        — Single-page overview
""")

  PM2.5 Visualisation Suite
  Loaded : XGBoost
  Loaded : LightGBM
  Loaded : RandomForest
  Loaded : GradientBoosting
  Loaded : CatBoost

  Generating plots …
  Saved → 01_metrics_comparison.png
  Saved → 02_scatter_predicted_vs_actual.png
  Saved → 03_residual_distribution.png
  Saved → 04_overfit_gap_heatmap.png
  Saved → 05_timeseries_actual_vs_predicted.png
  Saved → 06_feature_importance_xgboost.png
  Saved → 07_error_by_concentration_bin.png
  Saved → 08_radar_chart_metrics.png
  Saved → 09_residuals_vs_predicted.png
  Saved → 10_dashboard_summary.png

  All 10 plots saved to:
  C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports

  01_metrics_comparison.png       — R², MAE, RMSE across splits
  02_scatter_predicted_vs_actual.png — Predicted vs Actual scatter
  03_residual_distribution.png    — Residual histograms
  04_overfit_gap_heatmap.png      — Heatmap + gap bar
  05_timeseries_actual_vs_predicted.png — Time series overlay
  06_feature_importance_xgboost.png — To

In [2]:
"""
Feature Comparison Plots — final_ml_featured.parquet
=====================================================
Generates 6 publication-quality plots comparing features
in the dataset.

Saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\features\\

Run:
    python feature_plots.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F8F9FA",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.2,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
})

PALETTE    = ["#E63946","#F4A261","#2A9D8F","#457B9D","#6A0572",
              "#264653","#E9C46A","#A8DADC","#F1FAEE","#1D3557"]
SEASON_PAL = {"Winter":"#457B9D","Pre-Monsoon":"#F4A261",
              "Monsoon":"#2A9D8F","Post-Monsoon":"#E63946"}

def save(fig, name):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p.name}")

# ════════════════════════════════════════════
# LOAD
# ════════════════════════════════════════════
print("=" * 62)
print("  Feature Comparison Plots")
print("=" * 62)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")

print(f"  Loaded : {df.shape[0]:,} rows | {df.shape[1]} cols")
print(f"  Saving to: {REPORT_DIR}\n")

# ── Helper: pick columns that actually exist ───────────────────────────────
def pick(candidates, df=df):
    return [c for c in candidates if c in df.columns]

# ── Feature groups ─────────────────────────────────────────────────────────
AOD_COLS  = pick(["AOD_mean","AOD_max","AOD_p75","AOD_lag1","AOD_lag2","AOD_roll3","AOD_roll7"])
MET_COLS  = pick(["SPEED_mean","TLML_mean","TLML_max","moisture_max","rain_3day",
                   "rain_lag1","PRECTOTLAND","RHOA_mean","QV2M_mean","T2M_mean",
                   "WS10M_mean","PS_mean","U10M_mean","V10M_mean"])
STAT_COLS = pick(["station_pm_mean","station_pm_std","station_month_pm"])
LAG_COLS  = pick(["pm25_lag1","pm25_lag2","pm25_lag3","pm25_lag7","pm25_lag14"])
ROLL_COLS = pick(["pm25_roll3","pm25_roll7","pm25_roll14","pm25_roll30"])
EWM_COLS  = pick(["pm25_ewm7","pm25_ewm14"])

MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

# ════════════════════════════════════════════
# PLOT 1 — AOD vs PM2.5 Scatter Grid
#           One panel per AOD feature, coloured by season
# ════════════════════════════════════════════
print("[1] AOD vs PM2.5 scatter grid …")

aod_plot = AOD_COLS[:6] if len(AOD_COLS) >= 6 else AOD_COLS
ncols    = 3
nrows    = (len(aod_plot) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
fig.suptitle("AOD Features vs PM2.5 — Correlation Scatter (coloured by Season)",
             fontsize=15, fontweight="bold")
axes = axes.flatten()

for ax, col in zip(axes, aod_plot):
    tmp = df[["pm25", col]].replace(-1, np.nan).dropna()
    if "season" in df.columns:
        tmp["season"] = df.loc[tmp.index, "season"]
        for s, grp in tmp.groupby("season"):
            ax.scatter(grp[col], grp["pm25"],
                       alpha=0.18, s=6, label=s,
                       color=SEASON_PAL.get(s, "#888"), rasterized=True)
        ax.legend(fontsize=7, markerscale=3, framealpha=0.7)
    else:
        ax.scatter(tmp[col], tmp["pm25"], alpha=0.2, s=6,
                   color="#457B9D", rasterized=True)

    # Regression line
    if len(tmp) > 10:
        slope, intercept, r, p, _ = stats.linregress(tmp[col], tmp["pm25"])
        xs = np.linspace(tmp[col].min(), tmp[col].max(), 200)
        ax.plot(xs, slope * xs + intercept, color="#E63946", lw=2)
        ax.set_title(f"{col}\n r = {r:.3f}  (p {'< 0.001' if p < 0.001 else f'= {p:.3f}'})",
                     fontweight="bold", fontsize=10)

    ax.set_xlabel(col)
    ax.set_ylabel("PM2.5 (µg/m³)")

for ax in axes[len(aod_plot):]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "F01_aod_vs_pm25_scatter.png")

# ════════════════════════════════════════════
# PLOT 2 — PM2.5 Distribution by Month & Season
#           Box plots + violin
# ════════════════════════════════════════════
print("[2] PM2.5 seasonal & monthly distribution …")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("PM2.5 Distribution by Month and Season", fontsize=15, fontweight="bold")

# Monthly box plot
month_order_present = [m for m in MONTH_ORDER if m in df["month_name"].values]
sns.boxplot(data=df, x="month_name", y="pm25",
            order=month_order_present,
            palette=sns.color_palette("coolwarm", 12),
            width=0.6, linewidth=0.8,
            flierprops=dict(marker="o", ms=2, alpha=0.3),
            ax=axes[0])
axes[0].set_title("Monthly PM2.5 Distribution", fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("PM2.5 (µg/m³)")
axes[0].tick_params(axis="x", rotation=0)

# Monthly median line overlay
monthly_med = df.groupby("month_name")["pm25"].median().reindex(month_order_present)
ax_twin = axes[0].twinx()
ax_twin.plot(range(len(monthly_med)), monthly_med.values,
             "k-o", lw=2, ms=5, label="Median")
ax_twin.set_ylabel("Median PM2.5", color="black")
ax_twin.spines["top"].set_visible(False)

# Seasonal violin — drawn manually to avoid seaborn palette bugs in newer versions
if "season" in df.columns:
    season_order = [s for s in ["Winter","Pre-Monsoon","Monsoon","Post-Monsoon"]
                    if s in df["season"].values]
    for i, s in enumerate(season_order):
        sub = df.loc[df["season"] == s, "pm25"].dropna().values
        if len(sub) < 10:
            continue
        parts = axes[1].violinplot(sub, positions=[i], widths=0.6,
                                   showmedians=True, showextrema=True)
        col = SEASON_PAL.get(s, "#888")
        for pc in parts.get("bodies", []):
            pc.set_facecolor(col)
            pc.set_alpha(0.75)
        for key in ["cmedians", "cmins", "cmaxes", "cbars"]:
            if key in parts:
                parts[key].set_color(col)
                parts[key].set_linewidth(1.5)
    axes[1].set_xticks(range(len(season_order)))
    axes[1].set_xticklabels(season_order)
    axes[1].set_title("Seasonal PM2.5 Distribution (Violin)", fontweight="bold")
    axes[1].set_xlabel("Season")
    axes[1].set_ylabel("PM2.5 (µg/m³)")
else:
    sns.boxplot(data=df, x="year", y="pm25", ax=axes[1],
                palette="coolwarm")
    axes[1].set_title("PM2.5 Distribution by Year", fontweight="bold")

plt.tight_layout()
save(fig, "F02_pm25_seasonal_monthly.png")

# ════════════════════════════════════════════
# PLOT 3 — Correlation Heatmap
#           PM2.5 + AOD + Top Met features
# ════════════════════════════════════════════
print("[3] Feature correlation heatmap …")

corr_cols = pick(["pm25"] + AOD_COLS[:5] + MET_COLS[:8] + STAT_COLS[:3])
corr_df   = df[corr_cols].replace(-1, np.nan).copy()

# Drop cols with >60% missing
corr_df   = corr_df.loc[:, corr_df.isnull().mean() < 0.6]
corr_mat  = corr_df.corr()

mask = np.triu(np.ones_like(corr_mat, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 12))
fig.suptitle("Feature Correlation Matrix (Pearson)", fontsize=15, fontweight="bold")

sns.heatmap(corr_mat, mask=mask, ax=ax,
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7.5},
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Pearson r", "shrink": 0.8})

ax.set_title("Lower triangle: Pearson r — red = positive, blue = negative",
             fontsize=10, style="italic", pad=6)
plt.tight_layout()
save(fig, "F03_feature_correlation_heatmap.png")

# ════════════════════════════════════════════
# PLOT 4 — Meteorological Features vs PM2.5
#           2-row grid: scatter + regression per met variable
# ════════════════════════════════════════════
print("[4] Meteorological features vs PM2.5 …")

met_plot = MET_COLS[:8]
if not met_plot:
    print("  [SKIP] No met columns found.")
else:
    ncols = 4
    nrows = (len(met_plot) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    fig.suptitle("Meteorological Features vs PM2.5", fontsize=15, fontweight="bold")
    axes = axes.flatten()

    for ax, col in zip(axes, met_plot):
        tmp = df[["pm25", col]].dropna()
        # Hex bin — handles large N better than scatter
        hb = ax.hexbin(tmp[col], tmp["pm25"],
                       gridsize=35, cmap="YlOrRd",
                       mincnt=1, linewidths=0.2)
        plt.colorbar(hb, ax=ax, label="Count", pad=0.02)

        if len(tmp) > 10:
            slope, intercept, r, p, _ = stats.linregress(tmp[col], tmp["pm25"])
            xs = np.linspace(tmp[col].min(), tmp[col].max(), 200)
            ax.plot(xs, slope * xs + intercept, "b-", lw=1.8)
            ax.set_title(f"{col}\n r = {r:.3f}", fontweight="bold", fontsize=10)

        ax.set_xlabel(col)
        ax.set_ylabel("PM2.5 (µg/m³)")

    for ax in axes[len(met_plot):]:
        ax.set_visible(False)

    plt.tight_layout()
    save(fig, "F04_met_features_vs_pm25.png")

# ════════════════════════════════════════════
# PLOT 5 — PM2.5 Lag & Rolling Feature Importance
#           Correlation of lag/roll features with pm25
#           + autocorrelation structure
# ════════════════════════════════════════════
print("[5] Lag & rolling feature correlation …")

# Build lag cols if not already present
grp = df.groupby("station_name")["pm25"]
for lag in [1, 2, 3, 7, 14]:
    c = f"pm25_lag{lag}"
    if c not in df.columns:
        df[c] = grp.shift(lag)
for w in [3, 7, 14, 30]:
    c = f"pm25_roll{w}"
    if c not in df.columns:
        df[c] = grp.shift(1).transform(lambda x: x.rolling(w, min_periods=1).mean())
for sp, c in [(7,"pm25_ewm7"),(14,"pm25_ewm14")]:
    if c not in df.columns:
        df[c] = grp.shift(1).transform(lambda x: x.ewm(span=sp, min_periods=1).mean())

lag_roll_cols = pick(
    ["pm25_lag1","pm25_lag2","pm25_lag3","pm25_lag7","pm25_lag14",
     "pm25_roll3","pm25_roll7","pm25_roll14","pm25_roll30",
     "pm25_ewm7","pm25_ewm14"]
)

corr_vals = {c: df[["pm25", c]].dropna().corr().iloc[0, 1]
             for c in lag_roll_cols}
corr_s = pd.Series(corr_vals).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("PM2.5 Temporal Features — Autocorrelation Analysis",
             fontsize=15, fontweight="bold")

# Bar chart — correlation with pm25
colors_bar = ["#E63946" if "lag" in c else "#2A9D8F" if "roll" in c else "#F4A261"
               for c in corr_s.index]
bars = axes[0].barh(corr_s.index[::-1], corr_s.values[::-1],
                    color=colors_bar[::-1], edgecolor="white", linewidth=0.7)
for bar, v in zip(bars, corr_s.values[::-1]):
    axes[0].text(v + 0.003, bar.get_y() + bar.get_height() / 2,
                 f"{v:.3f}", va="center", fontsize=9)
axes[0].set_xlim(0, 1.05)
axes[0].set_title("Pearson r with PM2.5\n(red=lag | green=rolling | orange=EWM)",
                  fontweight="bold")
axes[0].set_xlabel("Correlation coefficient")

# Line plot — lag decay curve
lag_nums  = [1, 2, 3, 7, 14]
lag_corrs = [corr_vals.get(f"pm25_lag{l}", np.nan) for l in lag_nums]
roll_nums  = [3, 7, 14, 30]
roll_corrs = [corr_vals.get(f"pm25_roll{w}", np.nan) for w in roll_nums]

axes[1].plot(lag_nums,  lag_corrs,  "o-", color="#E63946", lw=2, ms=7, label="Lag")
axes[1].plot(roll_nums, roll_corrs, "s-", color="#2A9D8F", lw=2, ms=7, label="Rolling mean")
for sp, col, mk in [(7,"#F4A261","^"),(14,"#6A0572","D")]:
    v = corr_vals.get(f"pm25_ewm{sp}", np.nan)
    if not np.isnan(v):
        axes[1].scatter([sp], [v], color=col, marker=mk, s=100, zorder=5,
                        label=f"EWM-{sp}")
axes[1].set_title("Autocorrelation Decay Curve", fontweight="bold")
axes[1].set_xlabel("Lag / Window size (days)")
axes[1].set_ylabel("Pearson r with PM2.5")
axes[1].set_ylim(0.5, 1.0)
axes[1].legend(fontsize=9)

plt.tight_layout()
save(fig, "F05_lag_rolling_autocorrelation.png")

# ════════════════════════════════════════════
# PLOT 6 — Station-level Feature Comparison
#           Per-station PM2.5 mean, AOD mean, met mean
#           Sorted by PM2.5 mean, top-20 stations
# ════════════════════════════════════════════
print("[6] Station-level feature comparison …")

stat_agg = df.groupby("station_name").agg(
    pm25_mean   = ("pm25",    "mean"),
    pm25_std    = ("pm25",    "std"),
    pm25_median = ("pm25",    "median"),
    count       = ("pm25",    "count"),
).reset_index()

# AOD mean if available
if "AOD_mean" in df.columns:
    aod_agg = df.replace({"AOD_mean": {-1: np.nan}}) \
                .groupby("station_name")["AOD_mean"].mean().reset_index()
    aod_agg.columns = ["station_name", "aod_mean"]
    stat_agg = stat_agg.merge(aod_agg, on="station_name", how="left")

# Met feature if available
met_feat = next((c for c in ["SPEED_mean","TLML_mean","moisture_max"] if c in df.columns), None)
if met_feat:
    met_agg = df.groupby("station_name")[met_feat].mean().reset_index()
    met_agg.columns = ["station_name", "met_val"]
    stat_agg = stat_agg.merge(met_agg, on="station_name", how="left")

stat_agg = stat_agg.sort_values("pm25_mean", ascending=False).head(20).reset_index(drop=True)
labels   = stat_agg["station_name"].str[:14]   # truncate long names

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Station-Level Feature Comparison — Top 20 Stations by PM2.5",
             fontsize=15, fontweight="bold")

# [0,0] PM2.5 mean + std error bars
ax0 = fig.add_subplot(gs[0, 0])
colors_st = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(stat_agg)))
bars = ax0.bar(range(len(stat_agg)), stat_agg["pm25_mean"],
               yerr=stat_agg["pm25_std"], capsize=3,
               color=colors_st, edgecolor="white", linewidth=0.6,
               error_kw={"elinewidth": 1, "ecolor": "#555"})
ax0.set_xticks(range(len(stat_agg)))
ax0.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax0.set_title("Mean PM2.5 ± Std Dev per Station", fontweight="bold")
ax0.set_ylabel("PM2.5 (µg/m³)")

# [0,1] AOD mean per station
ax1 = fig.add_subplot(gs[0, 1])
if "aod_mean" in stat_agg.columns:
    ax1.bar(range(len(stat_agg)), stat_agg["aod_mean"],
            color="#457B9D", edgecolor="white", linewidth=0.6, alpha=0.85)
    ax1.set_xticks(range(len(stat_agg)))
    ax1.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax1.set_title("Mean AOD per Station", fontweight="bold")
    ax1.set_ylabel("AOD")
else:
    ax1.text(0.5, 0.5, "AOD_mean not available", ha="center", va="center",
             transform=ax1.transAxes, fontsize=13, color="gray")
    ax1.set_title("AOD (not available)", fontweight="bold")

# [1,0] PM2.5 median per station
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(range(len(stat_agg)), stat_agg["pm25_median"],
        color="#2A9D8F", edgecolor="white", linewidth=0.6, alpha=0.85)
ax2.set_xticks(range(len(stat_agg)))
ax2.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax2.set_title("Median PM2.5 per Station", fontweight="bold")
ax2.set_ylabel("PM2.5 (µg/m³)")

# [1,1] PM2.5 vs AOD dual-axis, or met variable
ax3 = fig.add_subplot(gs[1, 1])
x = range(len(stat_agg))
ax3.bar(x, stat_agg["pm25_mean"], color="#E63946", alpha=0.7,
        label="PM2.5 mean", edgecolor="white", linewidth=0.6)
ax3.set_ylabel("PM2.5 (µg/m³)", color="#E63946")
ax3.tick_params(axis="y", labelcolor="#E63946")

if "aod_mean" in stat_agg.columns:
    ax3b = ax3.twinx()
    ax3b.plot(x, stat_agg["aod_mean"], "o-",
              color="#457B9D", lw=2, ms=5, label="AOD mean")
    ax3b.set_ylabel("AOD", color="#457B9D")
    ax3b.tick_params(axis="y", labelcolor="#457B9D")
    ax3b.spines["top"].set_visible(False)
    lines1, labs1 = ax3.get_legend_handles_labels()
    lines2, labs2 = ax3b.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labs1 + labs2, fontsize=8, loc="upper right")
elif "met_val" in stat_agg.columns:
    ax3b = ax3.twinx()
    ax3b.plot(x, stat_agg["met_val"], "s-",
              color="#2A9D8F", lw=2, ms=5, label=met_feat)
    ax3b.set_ylabel(met_feat, color="#2A9D8F")
    ax3b.tick_params(axis="y", labelcolor="#2A9D8F")
    ax3b.spines["top"].set_visible(False)

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax3.set_title("PM2.5 vs AOD — Dual Axis per Station", fontweight="bold")

plt.tight_layout()
save(fig, "F06_station_feature_comparison.png")

# ════════════════════════════════════════════
# DONE
# ════════════════════════════════════════════
print("\n" + "=" * 62)
print(f"  All 6 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 62)
print("""
  F01_aod_vs_pm25_scatter.png         — AOD features vs PM2.5
  F02_pm25_seasonal_monthly.png       — Monthly & seasonal distribution
  F03_feature_correlation_heatmap.png — Full correlation matrix
  F04_met_features_vs_pm25.png        — Meteorological features vs PM2.5
  F05_lag_rolling_autocorrelation.png — Lag/roll autocorrelation decay
  F06_station_feature_comparison.png  — Station-level PM2.5 + AOD
""")

  Feature Comparison Plots
  Loaded : 66,617 rows | 64 cols
  Saving to: C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features

[1] AOD vs PM2.5 scatter grid …
  Saved → F01_aod_vs_pm25_scatter.png
[2] PM2.5 seasonal & monthly distribution …
  Saved → F02_pm25_seasonal_monthly.png
[3] Feature correlation heatmap …
  Saved → F03_feature_correlation_heatmap.png
[4] Meteorological features vs PM2.5 …
  Saved → F04_met_features_vs_pm25.png
[5] Lag & rolling feature correlation …
  Saved → F05_lag_rolling_autocorrelation.png
[6] Station-level feature comparison …
  Saved → F06_station_feature_comparison.png

  All 6 plots saved to:
  C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features

  F01_aod_vs_pm25_scatter.png         — AOD features vs PM2.5
  F02_pm25_seasonal_monthly.png       — Monthly & seasonal distribution
  F03_feature_correlation_heatmap.png — Full correlation matrix
  F04_met_features_vs_pm25.png        — Meteorological features vs PM2.5
  F

In [11]:
"""
Comprehensive Model Analysis — PM2.5 AOD Study
===============================================
Tests overfitting, compares all models, and visualises:

  PLOT 01 — Overfitting diagnosis  (train vs test R² / MAE / RMSE)
  PLOT 02 — Predicted vs Actual scatter  (all models)
  PLOT 03 — Time-series comparison  (first 500 test samples)
  PLOT 04 — Residual distributions
  PLOT 05 — MAE by PM2.5 concentration bin
  PLOT 06 — Residuals by season
  PLOT 07 — Top-20 feature importances  (all models)
  PLOT 08 — Feature correlation heatmap  (top-25 features)
  PLOT 09 — PM2.5 vs key features  (scatter + regression)
  PLOT 10 — Ensemble weight breakdown + metric summary table
  PLOT 11 — Clean metric bar chart overview

Saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\withoutlag\\

Run:
    python model_analysis_plots.py
"""

import warnings, pathlib, os
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.metrics       import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\models")
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\withoutlag")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DATE = pd.Timestamp("2024-07-01")

C = {
    "RandomForest"    : "#2A9D8F",
    "XGBoost"         : "#E63946",
    "LightGBM"        : "#F4A261",
    "GradientBoosting": "#457B9D",
    "Ensemble"        : "#1D3557",
}
SEASON_PAL = {
    "Winter"      : "#1D6FA4",
    "Pre-Monsoon" : "#F4A261",
    "Monsoon"     : "#2A9D8F",
    "Post-Monsoon": "#E76F51",
}

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F0F4F8",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.4,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 10,
    "xtick.labelsize"  : 8.5,
    "ytick.labelsize"  : 8.5,
    "legend.framealpha": 0.9,
    "legend.fontsize"  : 8.5,
})

def save(fig, name, dpi=160):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓  {name}")

def metrics(y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    return dict(R2=r2, MAE=mae, RMSE=rmse, Bias=bias)


# ═══════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("  Loading data ...")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
print(f"  Shape : {df.shape}")

# keep original season strings for plot 06 BEFORE encoding
df_season_str = df["season"].copy() if "season" in df.columns else None

# encode season to numeric so models can consume it
if "season" in df.columns:
    df["season"] = LabelEncoder().fit_transform(df["season"].astype(str))

# columns to exclude — lat / lon / season are KEPT because
# the saved models were trained with them
EXCLUDE = {
    "pm25", "date", "station_name", "station_id",
    "latitude", "longitude",
    "aod_quality_cat", "station_month_aod_n",
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE
    and df[c].dtype in ["float64", "float32", "int64", "int32", "bool", "uint8"]
]
print(f"  Features : {len(feature_cols)}")

# temporal split
train_df = df[df["date"] <  SPLIT_DATE].copy()
test_df  = df[df["date"] >= SPLIT_DATE].copy()

X_train = train_df[feature_cols].astype(float)
y_train = train_df["pm25"].values
X_test  = test_df[feature_cols].astype(float)
y_test  = test_df["pm25"].values

print(f"  Train : {X_train.shape}  |  Test : {X_test.shape}")


# ═══════════════════════════════════════════════════════════
# 2. LOAD MODELS
# ═══════════════════════════════════════════════════════════
print("\n  Loading models ...")

PKL = {
    "RandomForest"    : "rf_model.pkl",
    "XGBoost"         : "xgb_model.pkl",
    "LightGBM"        : "lgb_model.pkl",
    "GradientBoosting": "gb_model.pkl",
}
models = {}
for name, fname in PKL.items():
    p = MODEL_DIR / fname
    if p.exists():
        models[name] = joblib.load(p)
        print(f"    Loaded : {name}")
    else:
        print(f"    MISSING: {fname} — skipping")

if not models:
    raise RuntimeError("No models found in MODEL_DIR. Check path.")

# align column order to what the models expect
first_model = list(models.values())[0]
if hasattr(first_model, "feature_names_in_"):
    model_feat_cols = list(first_model.feature_names_in_)
    for c in model_feat_cols:
        if c not in X_train.columns:
            print(f"    Adding missing column '{c}' filled with 0")
            X_train[c] = 0.0
            X_test[c]  = 0.0
    X_train      = X_train[model_feat_cols]
    X_test       = X_test[model_feat_cols]
    feature_cols = model_feat_cols
    print(f"    Column order aligned  ({len(feature_cols)} features)")

# predictions
tr_pred, te_pred = {}, {}
for name, m in models.items():
    tr_pred[name] = np.maximum(m.predict(X_train), 0)
    te_pred[name] = np.maximum(m.predict(X_test),  0)

# ensemble
ens_w_path  = MODEL_DIR / "ensemble_weights.pkl"
ens_weights = joblib.load(ens_w_path) if ens_w_path.exists() else None

if ens_weights:
    key_map = {"rf":"RandomForest", "xgb":"XGBoost",
               "lgb":"LightGBM",    "gb":"GradientBoosting"}
    ens_te = np.zeros(len(y_test))
    ens_tr = np.zeros(len(y_train))
    for k, w in ens_weights.items():
        nm = key_map.get(k, k)
        if nm in te_pred:
            ens_te += w * te_pred[nm]
            ens_tr += w * tr_pred[nm]
    te_pred["Ensemble"] = np.maximum(ens_te, 0)
    tr_pred["Ensemble"] = np.maximum(ens_tr, 0)
    print("    Ensemble predictions computed")

model_names = list(te_pred.keys())

# metrics
tr_metrics = {n: metrics(y_train, tr_pred[n]) for n in model_names}
te_metrics = {n: metrics(y_test,  te_pred[n]) for n in model_names}

print("\n  OVERFITTING CHECK")
print(f"  {'Model':<22} {'Train R2':>9} {'Test R2':>9} {'Delta R2':>10}  "
      f"{'Train MAE':>10} {'Test MAE':>9} {'Delta MAE':>10}")
print(f"  {'-'*85}")
for n in model_names:
    tr, te   = tr_metrics[n], te_metrics[n]
    gap_r2   = tr["R2"]  - te["R2"]
    gap_mae  = te["MAE"] - tr["MAE"]
    flag     = "  << OVERFIT" if gap_r2 > 0.08 else ""
    print(f"  {n:<22} {tr['R2']:>9.4f} {te['R2']:>9.4f} {gap_r2:>10.4f}  "
          f"{tr['MAE']:>10.2f} {te['MAE']:>9.2f} {gap_mae:>10.2f}{flag}")


# ═══════════════════════════════════════════════════════════
# PLOT 01 — Overfitting Diagnosis
# ═══════════════════════════════════════════════════════════
print("\n[Plot 01] Overfitting diagnosis ...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Plot 01 — Overfitting Diagnosis: Train vs Test Performance",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.02)

for ax, (met, label) in zip(axes, [
    ("R2",   "R2  (higher = better)"),
    ("MAE",  "MAE ug/m3  (lower = better)"),
    ("RMSE", "RMSE ug/m3  (lower = better)"),
]):
    names = [n for n in model_names if n != "Ensemble"]
    tr_v  = [tr_metrics[n][met] for n in names]
    te_v  = [te_metrics[n][met] for n in names]
    x, w  = np.arange(len(names)), 0.32

    ax.bar(x - w/2, tr_v, w, label="Train", color="#2A9D8F", alpha=0.85, edgecolor="white")
    ax.bar(x + w/2, te_v, w, label="Test",  color="#E63946", alpha=0.85, edgecolor="white")

    rng = max(max(tr_v), max(te_v)) - min(min(tr_v), min(te_v))
    rng = rng if rng > 0 else 1
    for i, (tv, ev) in enumerate(zip(tr_v, te_v)):
        gap = tv - ev if met == "R2" else ev - tv
        col = "#C0392B" if abs(gap) > (0.08 if met == "R2" else 5) else "#2C3E50"
        ax.text(x[i], max(tv, ev) + rng * 0.03,
                f"D{abs(gap):.3f}", ha="center", fontsize=7.5,
                color=col, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=18, ha="right")
    ax.set_ylabel(label)
    ax.set_title(met, fontweight="bold", color="#1D3557")
    ax.legend()
    if met == "R2":
        ax.set_ylim(min(min(tr_v), min(te_v)) * 0.96, 1.01)

plt.tight_layout()
save(fig, "plot01_overfitting_diagnosis.png")


# ═══════════════════════════════════════════════════════════
# PLOT 02 — Predicted vs Actual Scatter
# ═══════════════════════════════════════════════════════════
print("[Plot 02] Predicted vs Actual scatter ...")

ncols = 3
nrows = (len(model_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5.5*nrows))
axes = np.array(axes).flatten()
fig.suptitle("Plot 02 — Predicted vs Actual (Test Set)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

for i, name in enumerate(model_names):
    ax  = axes[i]
    yp  = te_pred[name]
    col = C.get(name, "#888")
    lim = max(float(y_test.max()), float(yp.max())) * 1.04

    ax.plot([0, lim], [0, lim], "k--", lw=1.6, label="Perfect fit", zorder=10)
    ax.fill_between([0, lim], [-15, lim-15], [15, lim+15], alpha=0.07, color=col)
    ax.scatter(y_test, yp, alpha=0.15, s=6, color=col, rasterized=True)

    sl, ic, r, *_ = stats.linregress(y_test, yp)
    xs = np.linspace(0, lim, 300)
    ax.plot(xs, sl*xs + ic, color=col, lw=2.0, alpha=0.9, label=f"Fit  r={r:.3f}")

    m = te_metrics[name]
    ax.text(0.04, 0.95,
            f"R2={m['R2']:.3f}\nMAE={m['MAE']:.1f}\nRMSE={m['RMSE']:.1f}\nBias={m['Bias']:.1f}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=col, alpha=0.9))

    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Actual PM2.5 (ug/m3)")
    ax.set_ylabel("Predicted PM2.5 (ug/m3)")
    ax.set_title(name, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=7.5)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot02_predicted_vs_actual.png")


# ═══════════════════════════════════════════════════════════
# PLOT 03 — Time Series Comparison
# ═══════════════════════════════════════════════════════════
print("[Plot 03] Time-series comparison ...")

N = min(500, len(y_test))
fig, ax = plt.subplots(figsize=(22, 6))
fig.suptitle("Plot 03 — Time Series: Actual vs All Models (first 500 test samples)",
             fontsize=13, fontweight="bold", color="#1D3557")

ax.plot(range(N), y_test[:N], color="black", lw=2.2, label="Actual", zorder=10)
styles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
for i, name in enumerate(model_names):
    ax.plot(range(N), te_pred[name][:N],
            color=C.get(name, "#888"),
            lw=2.2 if name == "Ensemble" else 1.5,
            ls=styles[i % len(styles)],
            alpha=0.85, label=name, zorder=5+i)

ax.set_xlabel("Sample index (test set)")
ax.set_ylabel("PM2.5 (ug/m3)")
ax.legend(ncol=3, fontsize=8.5, loc="upper right")
ax.set_xlim(0, N - 1)
plt.tight_layout()
save(fig, "plot03_timeseries_comparison.png")


# ═══════════════════════════════════════════════════════════
# PLOT 04 — Residual Distributions
# ═══════════════════════════════════════════════════════════
print("[Plot 04] Residual distributions ...")

ncols = 3
nrows = (len(model_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows))
axes = np.array(axes).flatten()
fig.suptitle("Plot 04 — Residual Distributions (Test Set)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

for i, name in enumerate(model_names):
    ax    = axes[i]
    resid = y_test - te_pred[name]
    col   = C.get(name, "#888")

    ax.hist(resid, bins=60, color=col, alpha=0.75, edgecolor="white", density=True)
    xr = np.linspace(float(resid.min()), float(resid.max()), 300)
    ax.plot(xr, stats.gaussian_kde(resid)(xr), color="black", lw=2.0)
    ax.axvline(0,            color="red",  lw=1.8, ls="--", label="Zero")
    ax.axvline(resid.mean(), color="navy", lw=1.5, ls=":",
               label=f"mean={resid.mean():.1f}")

    ax.set_xlabel("Residual (Actual - Predicted) ug/m3")
    ax.set_ylabel("Density")
    ax.set_title(name, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=8)

    _, p_sw = stats.shapiro(resid[:min(3000, len(resid))])
    ax.text(0.97, 0.95, f"Shapiro p={p_sw:.3f}",
            transform=ax.transAxes, fontsize=7.5, ha="right", va="top", color="#555")

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot04_residual_distributions.png")


# ═══════════════════════════════════════════════════════════
# PLOT 05 — MAE by PM2.5 Concentration Bin
# ═══════════════════════════════════════════════════════════
print("[Plot 05] MAE by concentration bin ...")

bins_e   = [0, 30, 60, 100, 150, 200, 300, 700]
bin_labs = ["0-30", "30-60", "60-100", "100-150", "150-200", "200-300", "300+"]
bin_col  = pd.cut(y_test, bins=bins_e, labels=bin_labs, right=False)
counts_b = [(bin_col == lb).sum() for lb in bin_labs]

fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Plot 05 — MAE by PM2.5 Concentration Bin",
             fontsize=13, fontweight="bold", color="#1D3557")

x = np.arange(len(bin_labs))
w = 0.80 / len(model_names)
for j, name in enumerate(model_names):
    mae_bins = [
        np.mean(np.abs(y_test[bin_col == lb] - te_pred[name][bin_col == lb]))
        if (bin_col == lb).sum() > 0 else 0
        for lb in bin_labs
    ]
    offset = (j - len(model_names) / 2) * w + w / 2
    ax.bar(x + offset, mae_bins, w,
           label=name, color=C.get(name, "#888"),
           alpha=0.85, edgecolor="white", linewidth=0.4)

ax2 = ax.twinx()
ax2.plot(x, counts_b, "ko-", lw=1.8, ms=5, label="Sample count")
ax2.set_ylabel("Sample count", fontsize=9)
ax2.spines["top"].set_visible(False)

ax.set_xticks(x)
ax.set_xticklabels(bin_labs, rotation=20, ha="right")
ax.set_xlabel("PM2.5 Concentration Range (ug/m3)")
ax.set_ylabel("MAE (ug/m3)")
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, fontsize=8, ncol=3, loc="upper left")
plt.tight_layout()
save(fig, "plot05_mae_by_pm25_bin.png")


# ═══════════════════════════════════════════════════════════
# PLOT 06 — Residuals by Season
# ═══════════════════════════════════════════════════════════
# print("[Plot 06] Residuals by season ...")

# if df_season_str is not None:
#     test_season_str = df_season_str[df["date"] >= SPLIT_DATE].values
#     season_order    = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]

#     ncols = 3
#     nrows = (len(model_names) + ncols - 1) // ncols
#     fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows))
#     axes = np.array(axes).flatten()
#     fig.suptitle("Plot 06 — Residuals by Season (Test Set)",
#                  fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

#     for idx, name in enumerate(model_names):
#         ax    = axes[idx]
#         resid = y_test - te_pred[name]

#         s_valid = [s for s in season_order
#                    if s in test_season_str
#                    and (test_season_str == s).sum() > 1]

#         bp_arr = np.empty(len(s_valid), dtype=object)
#         for k, s in enumerate(s_valid):
#             bp_arr[k] = resid[test_season_str == s]

#         parts = ax.boxplot(
#             bp_arr,
#             positions=list(range(len(s_valid))),
#             widths=0.55, patch_artist=True,
#             showfliers=True,
#             flierprops=dict(marker="o", ms=2.5, alpha=0.25),
#             medianprops=dict(color="black", lw=2.0),
#             whiskerprops=dict(lw=1.2),
#             capprops=dict(lw=1.2),
#         )
#         for patch, s in zip(parts["boxes"], s_valid):
#             patch.set_facecolor(SEASON_PAL.get(s, "#888"))
#             patch.set_alpha(0.80)

#         ax.axhline(0, color="red", lw=1.6, ls="--", label="Zero error")
#         ax.set_xticks(range(len(s_valid)))
#         ax.set_xticklabels(s_valid, rotation=15, ha="right")
#         ax.set_ylabel("Residual (Actual-Pred) ug/m3")
#         ax.set_title(name, fontweight="bold", color="#1D3557")
#         ax.legend(fontsize=8)
#         for ki, arr in enumerate(bp_arr):
#             ax.text(ki, float(np.percentile(arr, 92)) + 1.5,
#                     f"m={arr.mean():.1f}", ha="center", fontsize=7.5, color="#333")

#     for j in range(idx+1, len(axes)):
#         axes[j].set_visible(False)

#     plt.tight_layout()
#     save(fig, "plot06_residuals_by_season.png")
# else:
#     print("  'season' not available — skipping plot 06")


# ═══════════════════════════════════════════════════════════
# PLOT 07 — Feature Importances
# ═══════════════════════════════════════════════════════════
print("[Plot 07] Feature importances ...")

base_models = {n: m for n, m in models.items()
               if hasattr(m, "feature_importances_")}

if base_models:
    n_bm  = len(base_models)
    fig, axes = plt.subplots(1, n_bm, figsize=(7*n_bm, 10))
    if n_bm == 1:
        axes = [axes]
    fig.suptitle("Plot 07 — Top-20 Feature Importances",
                 fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

    for ax, (name, m) in zip(axes, base_models.items()):
        fi = pd.Series(m.feature_importances_, index=feature_cols)\
               .sort_values(ascending=False).head(20)
        col  = C.get(name, "#888")
        bars = ax.barh(range(len(fi)), fi.values[::-1],
                       color=col, alpha=0.82, edgecolor="white")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi.index[::-1], fontsize=8)
        ax.set_xlabel("Importance")
        ax.set_title(name, fontweight="bold", color="#1D3557")
        for bar, v in zip(bars, fi.values[::-1]):
            ax.text(v + fi.values.max() * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f"{v:.4f}", va="center", fontsize=7)

    plt.tight_layout()
    save(fig, "plot07_feature_importances.png")


# ═══════════════════════════════════════════════════════════
# PLOT 08 — Feature Correlation Heatmap
# ═══════════════════════════════════════════════════════════
print("[Plot 08] Feature correlation heatmap ...")

tmp_tr = X_train.copy()
tmp_tr["pm25"] = y_train
corr_with_target = (
    tmp_tr.corr()["pm25"]
    .drop("pm25")
    .abs()
    .sort_values(ascending=False)
    .head(25)
)
top_feats = corr_with_target.index.tolist()
corr_mat  = X_train[top_feats].corr()

fig, ax = plt.subplots(figsize=(14, 12))
fig.suptitle("Plot 08 — Feature Correlation Heatmap (Top-25 by |r| with PM2.5)",
             fontsize=13, fontweight="bold", color="#1D3557")

sns.heatmap(corr_mat,
            mask=np.triu(np.ones_like(corr_mat, dtype=bool)),
            ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.4, linecolor="white",
            annot=True, fmt=".2f", annot_kws={"size": 6.5},
            cbar_kws={"shrink": 0.75})
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=7.5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7.5)
plt.tight_layout()
save(fig, "plot08_feature_correlation_heatmap.png")


# ═══════════════════════════════════════════════════════════
# PLOT 09 — PM2.5 vs Key Features
# ═══════════════════════════════════════════════════════════
print("[Plot 09] PM2.5 vs key features ...")

key_feats = [f for f in [
    "AOD_mean", "PBLH_mean", "SPEED_mean", "moisture_mean",
    "vent_coeff_mean", "inversion_proxy_mean", "PM_proxy", "trap_score",
] if f in X_train.columns][:8]

ncols_f = 4
nrows_f = (len(key_feats) + ncols_f - 1) // ncols_f
fig, axes = plt.subplots(nrows_f, ncols_f, figsize=(6*ncols_f, 5*nrows_f))
axes = np.array(axes).flatten()
fig.suptitle("Plot 09 — PM2.5 vs Key Features (Training Data)",
             fontsize=13, fontweight="bold", color="#1D3557", y=1.01)

for i, feat in enumerate(key_feats):
    ax  = axes[i]
    tmp = pd.DataFrame({feat: X_train[feat].values, "pm25": y_train})
    tmp = tmp.replace(-1, np.nan).dropna()
    if len(tmp) < 50:
        ax.set_visible(False)
        continue

    ax.scatter(tmp[feat], tmp["pm25"],
               alpha=0.12, s=5, color="#457B9D", rasterized=True)
    sl, ic, r, *_ = stats.linregress(tmp[feat], tmp["pm25"])
    xs = np.linspace(float(tmp[feat].min()), float(tmp[feat].max()), 200)
    ax.plot(xs, sl*xs + ic, color="#E63946", lw=2.2, label=f"r = {r:.3f}")
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel("PM2.5 (ug/m3)", fontsize=9)
    ax.set_title(feat, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot09_pm25_vs_features.png")


# ═══════════════════════════════════════════════════════════
# PLOT 10 — Summary Table + Ensemble Weights
# ═══════════════════════════════════════════════════════════
print("[Plot 10] Summary table + ensemble weights ...")

fig = plt.figure(figsize=(20, 9))
fig.suptitle("Plot 10 — Full Metrics Summary & Ensemble Composition",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)
gs     = gridspec.GridSpec(1, 2, figure=fig, wspace=0.40)
ax_tbl = fig.add_subplot(gs[0, 0])
ax_pie = fig.add_subplot(gs[0, 1])

cols_t = ["Model","Train R2","Test R2","Delta R2",
          "Train MAE","Test MAE","RMSE","Bias"]
rows   = []
for n in model_names:
    tr, te = tr_metrics[n], te_metrics[n]
    gap    = tr["R2"] - te["R2"]
    rows.append([
        n + (" (!)" if gap > 0.08 else ""),
        f"{tr['R2']:.4f}", f"{te['R2']:.4f}", f"{gap:.4f}",
        f"{tr['MAE']:.2f}", f"{te['MAE']:.2f}",
        f"{te['RMSE']:.2f}", f"{te['Bias']:.2f}",
    ])

ax_tbl.axis("off")
tbl = ax_tbl.table(cellText=rows, colLabels=cols_t,
                   cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 2.2)
for j in range(len(cols_t)):
    tbl[0, j].set_facecolor("#1D3557")
    tbl[0, j].set_text_props(color="white", fontweight="bold")
for i, (row, name) in enumerate(zip(rows, model_names)):
    gap = tr_metrics[name]["R2"] - te_metrics[name]["R2"]
    bg  = "#FDECEA" if gap > 0.08 else ("#F0F4F8" if i % 2 == 0 else "white")
    for j in range(len(cols_t)):
        tbl[i+1, j].set_facecolor(bg)

ax_tbl.set_title("Model Performance Summary  (!) = R2 gap > 0.08",
                  fontweight="bold", color="#1D3557", pad=12)

if ens_weights:
    key_map2 = {"rf":"RandomForest","xgb":"XGBoost",
                "lgb":"LightGBM","gb":"GradientBoosting"}
    labels_e = [key_map2.get(k, k) for k in ens_weights]
    vals_e   = list(ens_weights.values())
    colors_e = [C.get(l, "#888") for l in labels_e]
    _, _, autotexts = ax_pie.pie(
        vals_e, labels=labels_e, colors=colors_e,
        autopct="%1.1f%%", startangle=140, pctdistance=0.75,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax_pie.set_title("Ensemble Blend Weights (weighted by Test R2)",
                      fontweight="bold", color="#1D3557", pad=12)
else:
    ax_pie.text(0.5, 0.5, "ensemble_weights.pkl not found",
                ha="center", va="center", transform=ax_pie.transAxes,
                color="gray", fontsize=11)

plt.tight_layout()
save(fig, "plot10_summary_table_ensemble.png")


# ═══════════════════════════════════════════════════════════
# PLOT 11 — Metric Bar Chart Overview
# ═══════════════════════════════════════════════════════════
print("[Plot 11] Metric bar chart overview ...")

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle("Plot 11 — Test Set Metric Overview (All Models)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.02)

for ax, met in zip(axes, ["R2", "MAE", "RMSE", "Bias"]):
    vals  = [te_metrics[n][met] for n in model_names]
    cols  = [C.get(n, "#888")   for n in model_names]
    bars  = ax.bar(range(len(model_names)), vals,
                   color=cols, alpha=0.85, edgecolor="white", width=0.55)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=22, ha="right")
    ax.set_ylabel(met)
    ax.set_title(met, fontweight="bold", color="#1D3557")
    ax.axhline(0, color="black", lw=0.6, alpha=0.2)
    if met == "R2":
        ax.axhline(1.0, color="black", lw=0.8, ls=":", alpha=0.4)
    rng = max(vals) - min(vals) if max(vals) != min(vals) else 1
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + rng * 0.03,
                f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
save(fig, "plot11_metric_overview.png")


# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  All 11 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 60)
print("""
  plot01 — Overfitting diagnosis  (Train vs Test R2/MAE/RMSE)
  plot02 — Predicted vs Actual scatter  (per model)
  plot03 — Time-series: Actual vs all models  (500 samples)
  plot04 — Residual distributions with KDE
  plot05 — MAE by PM2.5 concentration bin
  plot06 — Residuals by season  (per model)
  plot07 — Top-20 feature importances  (per model)
  plot08 — Feature correlation heatmap  (top-25)
  plot09 — PM2.5 vs key features  (scatter + regression)
  plot10 — Full metrics table + ensemble weights
  plot11 — Clean metric bar chart overview
""")

  Loading data ...
  Shape : (66617, 63)
  Features : 57
  Train : (49918, 57)  |  Test : (16699, 57)

  Loading models ...
    Loaded : RandomForest
    Loaded : XGBoost
    Loaded : LightGBM
    Loaded : GradientBoosting
    Column order aligned  (57 features)
    Ensemble predictions computed

  OVERFITTING CHECK
  Model                   Train R2   Test R2   Delta R2   Train MAE  Test MAE  Delta MAE
  -------------------------------------------------------------------------------------
  RandomForest              0.8850    0.6924     0.1925        9.17     15.08       5.91  << OVERFIT
  XGBoost                   0.8020    0.7012     0.1009       12.74     14.97       2.23  << OVERFIT
  LightGBM                  0.7983    0.7049     0.0934       12.83     14.89       2.07  << OVERFIT
  GradientBoosting          0.8564    0.6913     0.1652       10.79     15.13       4.33  << OVERFIT
  Ensemble                  0.8420    0.7053     0.1367       11.23     14.84       3.60  << OVERFIT


In [6]:
"""
generate_plots.py  -  PM2.5 Model Diagnostic Plots (complete rewrite)
All metrics / importances / seasonal stats taken directly from training logs.
Works in .py scripts and Jupyter notebooks alike.
"""

# ══════════════════════════════════════════════════════════════════════════════
# 0.  PATHS
# ══════════════════════════════════════════════════════════════════════════════
MODELS_DIR  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\extras\models"
OUT_NOLAG   = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag"
OUT_WITHLAG = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withlag"
PARQUET_A   = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\v1\dataset_A_no_lags.parquet"
PARQUET_B   = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\v1\dataset_B_with_lags.parquet"

# ══════════════════════════════════════════════════════════════════════════════
# 1.  IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import joblib, pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

os.makedirs(OUT_NOLAG,   exist_ok=True)
os.makedirs(OUT_WITHLAG, exist_ok=True)

C4     = ["#4C72B0","#DD8452","#55A868","#C44E52"]
CENS   = "#7B2D8B"
PAL    = C4 + [CENS]
MKEYS  = ["rf","xgb","lgb","gb"]
FNAME  = {"rf":"Random Forest","xgb":"XGBoost","lgb":"LightGBM",
          "gb":"Grad. Boost","ensemble":"Ensemble"}

plt.rcParams.update({
    "figure.dpi":150,"savefig.dpi":150,
    "font.family":"DejaVu Sans","font.size":10,
    "axes.titlesize":12,"axes.labelsize":11,
    "axes.grid":True,"grid.alpha":0.3,
    "axes.spines.top":False,"axes.spines.right":False,
})

# ══════════════════════════════════════════════════════════════════════════════
# 2.  KNOWN VALUES FROM TRAINING LOGS
# ══════════════════════════════════════════════════════════════════════════════
METRICS = {
    "A": {"rf":      dict(r2=0.7073,mae=15.67,rmse=24.59),
          "xgb":     dict(r2=0.7214,mae=15.25,rmse=23.99),
          "lgb":     dict(r2=0.7200,mae=15.28,rmse=24.05),
          "gb":      dict(r2=0.7023,mae=15.65,rmse=24.80),
          "ensemble":dict(r2=0.7211,mae=15.25,rmse=24.00)},
    "B": {"rf":      dict(r2=0.8483,mae=10.40,rmse=17.70),
          "xgb":     dict(r2=0.8514,mae=10.37,rmse=17.52),
          "lgb":     dict(r2=0.8507,mae=10.44,rmse=17.56),
          "gb":      dict(r2=0.8480,mae=10.43,rmse=17.72),
          "ensemble":dict(r2=0.8522,mae=10.32,rmse=17.47)},
}
WEIGHTS = {
    "A":{"rf":0.248,"xgb":0.253,"lgb":0.253,"gb":0.246},
    "B":{"rf":0.250,"xgb":0.251,"lgb":0.250,"gb":0.250},
}
IMP_A = pd.Series({
    "station_month_pm":0.460822,"moisture_max":0.070034,"month_cos":0.065020,
    "station_pm_mean":0.040767,"moisture_mean":0.032889,"rain_lag1":0.020590,
    "rain_3day":0.020535,"TLML_mean":0.017386,"station_pm_std":0.016136,
    "vent_coeff_mean":0.015590,"AOD_x_vent":0.014204,"month_sin":0.012234,
    "SPEED_mean":0.011388,"doy_sin":0.011280,"doy_cos":0.011078,
    "SPEED_min":0.008748,"PBLH_morning_mean":0.007876,"rain_lag2":0.007773,
    "day_of_year":0.006883,"vent_coeff_min":0.006694,
}).sort_values(ascending=False)
IMP_B = pd.Series({
    "pm25_lag1":0.462272,"pm25_roll3":0.215564,"pm25_roll7":0.062298,
    "station_month_pm":0.056979,"rain_3day":0.008925,"pm25_lag2":0.008343,
    "rain_lag1":0.008311,"SPEED_mean":0.007994,"vent_coeff_mean":0.007204,
    "pm25_delta1":0.006267,"AOD_x_vent":0.005817,"SPEED_min":0.005107,
    "pm25_lag3":0.004856,"doy_sin":0.004832,"PRECTOT_sum":0.004724,
    "month_cos":0.004652,"moisture_max":0.004387,"vent_coeff_min":0.004275,
    "moisture_mean":0.004089,"doy_cos":0.004080,
}).sort_values(ascending=False)
SEASONAL = {
    "A":{"Winter":(0.6170,2828),"Monsoon":(0.4079,5593),"Post-monsoon":(0.7055,8314)},
    "B":{"Winter":(0.8156,2828),"Monsoon":(0.6403,5593),"Post-monsoon":(0.8412,8314)},
}
AOD_R2 = {
    "A":{"Cloud (AOD=-1)":(0.5477,9185),"Clear (AOD>0)":(0.6716,7550)},
    "B":{"Cloud (AOD=-1)":(0.7310,9185),"Clear (AOD>0)":(0.8335,7550)},
}
QUINTILE_BIAS = {
    "A":[("Q1 (low)",+8.41,3347),("Q2",+5.19,3348),("Q3",+2.29,3346),
         ("Q4",+0.16,3347),("Q5 (high)",-13.33,3347)],
    "B":[("Q1 (low)",+4.36,3347),("Q2",+2.84,3348),("Q3",+1.11,3346),
         ("Q4",-0.53,3347),("Q5 (high)",-7.61,3347)],
}
FEAT_A = [
    'n_pixels','AOD_mean','AOD_max','AOD_p75','AOD_count','AOD_coverage',
    'PBLH_mean','PBLH_min','PBLH_max','TLML_mean','TLML_max',
    'moisture_mean','moisture_max','SPEED_mean','SPEED_min','PRECTOT_sum',
    'vent_coeff_mean','vent_coeff_min','inversion_proxy_mean','inversion_proxy_max',
    'wind_dir_mean','PBLH_morning_mean','PBLH_morning_min',
    'AOD_missing','AOD_reliable','AOD_high',
    'aod_miss_winter','aod_miss_premonsoon','aod_miss_monsoon','aod_miss_postmonsoon',
    'station_aod_missing_rate','station_aod_quality','station_month_aod_clim',
    'day_of_year','day_of_week','year','is_weekend','season',
    'month_sin','month_cos','doy_sin','doy_cos',
    'PM_proxy','PM_proxy_max',
    'AOD_x_moisture','AOD_x_vent','AOD_x_vent_min','AOD_x_inversion','AOD_x_PBLH_min',
    'trap_score','is_stagnant',
    'rain_lag1','rain_lag2','is_rainy','post_rain1','post_rain2','rain_3day',
    'AOD_lag1','AOD_lag2','AOD_roll3','AOD_roll7',
    'station_pm_mean','station_pm_std','station_month_pm','proxy_anomaly','lat',
]
FEAT_B = FEAT_A + ['pm25_lag1','pm25_lag2','pm25_lag3',
                    'pm25_roll3','pm25_roll7','pm25_delta1']
TSUF  = {"A":"Dataset A — no PM2.5 lags","B":"Dataset B — with PM2.5 lags"}

# ══════════════════════════════════════════════════════════════════════════════
# 3.  LOAD DATA & MODELS
# ══════════════════════════════════════════════════════════════════════════════
def load_pkl(path):
    try:    return joblib.load(path)
    except: 
        with open(path,"rb") as f: return pickle.load(f)

print("\n-- Loading datasets --")
df_A = pd.read_parquet(PARQUET_A)
df_B = pd.read_parquet(PARQUET_B)
for df in [df_A, df_B]:
    if "latitude"  in df.columns and "lat" not in df.columns: df.rename(columns={"latitude":"lat"},  inplace=True)
    if "longitude" in df.columns and "lon" not in df.columns: df.rename(columns={"longitude":"lon"}, inplace=True)

TARGET = "pm25"
SPLIT  = "2024-06-30"

def prep(df, feat_cols):
    df = df.copy()
    if "date" not in df.columns:
        df = df.reset_index().rename(columns={"index":"date"})
    df["date"] = pd.to_datetime(df["date"])
    for c in feat_cols:
        if c not in df.columns: df[c] = 0
    tr = df[df["date"] <= SPLIT]
    te = df[df["date"] >  SPLIT]
    return tr, te

train_A, test_A = prep(df_A, FEAT_A)
train_B, test_B = prep(df_B, FEAT_B)
X_tr_A,y_tr_A = train_A[FEAT_A], train_A[TARGET]
X_te_A,y_te_A = test_A [FEAT_A], test_A [TARGET]
X_tr_B,y_tr_B = train_B[FEAT_B], train_B[TARGET]
X_te_B,y_te_B = test_B [FEAT_B], test_B [TARGET]
print(f"  A train={X_tr_A.shape} test={X_te_A.shape}")
print(f"  B train={X_tr_B.shape} test={X_te_B.shape}")

MODEL_FILES = {
    ("rf","A"):"random_forest_A_no_lags.pkl",   ("rf","B"):"random_forest_B_with_lags.pkl",
    ("xgb","A"):"xgboost_A_no_lags.pkl",         ("xgb","B"):"xgboost_B_with_lags.pkl",
    ("lgb","A"):"lightgbm_A_no_lags.pkl",         ("lgb","B"):"lightgbm_B_with_lags.pkl",
    ("gb","A"):"gradient_boosting_A_no_lags.pkl", ("gb","B"):"gradient_boosting_B_with_lags.pkl",
}
print("\n-- Loading models --")
MDL = {"A":{},"B":{}}
for suffix in ["A","B"]:
    for m in MKEYS:
        fp = os.path.join(MODELS_DIR, MODEL_FILES[(m,suffix)])
        try:
            MDL[suffix][m] = load_pkl(fp)
            print(f"  ok  {MODEL_FILES[(m,suffix)]}")
        except Exception as e:
            print(f"  FAIL {MODEL_FILES[(m,suffix)]}: {e}")

def align(mdl, X):
    if not hasattr(mdl,"feature_names_in_"): return X
    exp = list(mdl.feature_names_in_)
    for c in exp:
        if c not in X.columns: X = X.copy(); X[c] = 0
    return X[exp]

print("\n-- Predicting --")
PREDS = {"A":{},"B":{}}
for suf, Xtr, ytr, Xte in [("A",X_tr_A,y_tr_A,X_te_A),("B",X_tr_B,y_tr_B,X_te_B)]:
    for m, mdl in MDL[suf].items():
        PREDS[suf][m] = {"train":mdl.predict(align(mdl,Xtr)),
                          "test": mdl.predict(align(mdl,Xte))}
    w  = WEIGHTS[suf]
    PREDS[suf]["ensemble"] = {
        "train": sum(PREDS[suf][m]["train"]*w[m] for m in MDL[suf]),
        "test":  sum(PREDS[suf][m]["test"] *w[m] for m in MDL[suf]),
    }

# meta
def season_of(mo):
    return ("Winter" if mo in [12,1,2] else "Pre-monsoon" if mo in [3,4,5]
            else "Monsoon" if mo in [6,7,8,9] else "Post-monsoon")

def make_meta(tdf):
    d = tdf.copy().reset_index(drop=True)
    d["season"]   = pd.to_datetime(d["date"]).dt.month.map(season_of)
    aod = "AOD_mean"
    d["aod_flag"] = (np.where(d[aod].values<=0,"Cloud (AOD=-1)","Clear (AOD>0)")
                     if aod in d.columns else "Unknown")
    return d

META = {"A": make_meta(test_A), "B": make_meta(test_B)}

def calc(yt, yp):
    r2   = r2_score(yt,yp)
    mae  = mean_absolute_error(yt,yp)
    rmse = np.sqrt(mean_squared_error(yt,yp))
    return r2, mae, rmse

def savefig(fig, od, fn):
    p = os.path.join(od, fn)
    fig.savefig(p, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"  => {p}")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 01  Overfitting Diagnosis
# ══════════════════════════════════════════════════════════════════════════════
def plot01(od, suf, Xtr, ytr, Xte, yte):
    print(f"\n[01] {suf}")
    avail = list(MDL[suf].keys())
    fig, axes = plt.subplots(1,len(avail),figsize=(5*len(avail),4.8),sharey=False)
    if len(avail)==1: axes=[axes]
    for ax,m,col in zip(axes,avail,C4):
        r2_tr,mae_tr,_ = calc(ytr, PREDS[suf][m]["train"])
        r2_te,mae_te,_ = calc(yte, PREDS[suf][m]["test"])
        b1 = ax.bar("Train", r2_tr, color=col, alpha=1.0,  width=0.45)
        b2 = ax.bar("Test",  r2_te, color=col, alpha=0.55, width=0.45)
        for b,v in [(b1[0],r2_tr),(b2[0],r2_te)]:
            ax.text(b.get_x()+b.get_width()/2, v+0.012,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.set_ylim(0,1.08); ax.set_title(FNAME[m])
        ax.set_ylabel("R2" if ax is axes[0] else "")
        ax.annotate(f"MAE train {mae_tr:.1f}\nMAE test  {mae_te:.1f}",
                    xy=(0.97,0.06),xycoords="axes fraction",ha="right",fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3",fc="white",alpha=0.8))
    fig.suptitle(f"Overfitting Diagnosis -- {TSUF[suf]}",fontsize=13,fontweight="bold",y=1.01)
    savefig(fig, od, "plot01_overfitting_diagnosis.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 02  Predicted vs Actual
# ══════════════════════════════════════════════════════════════════════════════
def plot02(od, suf, yte):
    print(f"\n[02] {suf}")
    keys = list(MDL[suf].keys()) + ["ensemble"]
    ncols=3; nrows=(len(keys)+2)//3
    fig, axes = plt.subplots(nrows,ncols,figsize=(5.5*ncols,5*nrows))
    axes = np.array(axes).flatten()
    lim  = (0, float(yte.max())+15)
    for ax,m,col in zip(axes,keys,PAL):
        yp = PREDS[suf][m]["test"]
        r2,mae,rmse = calc(yte,yp)
        ax.scatter(yte,yp,s=6,alpha=0.22,color=col,rasterized=True)
        ax.plot(lim,lim,"k--",lw=1)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("Actual PM2.5 (ug/m3)"); ax.set_ylabel("Predicted PM2.5 (ug/m3)")
        ax.set_title(f"{FNAME[m]}\nR2={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    for ax in axes[len(keys):]: ax.set_visible(False)
    fig.suptitle(f"Predicted vs Actual -- {TSUF[suf]}",fontsize=13,fontweight="bold")
    savefig(fig, od, "plot02_predicted_vs_actual.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 03  Time-series
# ══════════════════════════════════════════════════════════════════════════════
def plot03(od, suf, yte, meta):
    print(f"\n[03] {suf}")
    ens   = PREDS[suf]["ensemble"]["test"]
    dates = pd.to_datetime(meta["date"].values)
    df_ts = (pd.DataFrame({"date":dates,"actual":yte.values,"pred":ens})
               .groupby("date").mean().reset_index())
    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(15,7),sharex=True)
    ax1.plot(df_ts["date"],df_ts["actual"],lw=1.2,label="Actual",color="#2c7bb6")
    ax1.plot(df_ts["date"],df_ts["pred"],  lw=1.2,label="Predicted",color="#d7191c",alpha=0.85)
    ax1.set_ylabel("PM2.5 (ug/m3)"); ax1.legend(framealpha=0.5)
    ax1.set_title("Daily-mean PM2.5: Actual vs Ensemble")
    res = df_ts["pred"]-df_ts["actual"]
    colors_bar = ["#d7191c" if v>=0 else "#2c7bb6" for v in res]
    ax2.bar(df_ts["date"],res,color=colors_bar,width=1,alpha=0.7)
    ax2.axhline(0,color="k",lw=0.8)
    ax2.set_ylabel("Residual (Pred - Actual)"); ax2.set_xlabel("Date")
    ax2.set_title("Daily residual")
    fig.suptitle(f"Time-series Comparison -- {TSUF[suf]}",fontsize=13,fontweight="bold")
    savefig(fig, od, "plot03_timeseries_comparison.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 04  Residual Distributions
# ══════════════════════════════════════════════════════════════════════════════
def plot04(od, suf, yte):
    print(f"\n[04] {suf}")
    keys = list(MDL[suf].keys()) + ["ensemble"]
    fig, axes = plt.subplots(1,len(keys),figsize=(4*len(keys),4.5),sharey=True)
    if len(keys)==1: axes=[axes]
    for ax,m,col in zip(axes,keys,PAL):
        res = PREDS[suf][m]["test"] - yte.values
        ax.hist(res,bins=60,color=col,alpha=0.75,edgecolor="white",lw=0.3)
        ax.axvline(0,color="k",lw=1.2,ls="--")
        ax.axvline(res.mean(),color="red",lw=1.2,ls="-",label=f"mean={res.mean():.1f}")
        ax.set_title(FNAME[m]); ax.set_xlabel("Residual (ug/m3)")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Count")
    fig.suptitle(f"Residual Distributions -- {TSUF[suf]}",fontsize=13,fontweight="bold")
    savefig(fig, od, "plot04_residual_distributions.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 05  MAE by PM2.5 bin
# ══════════════════════════════════════════════════════════════════════════════
def plot05(od, suf, yte):
    print(f"\n[05] {suf}")
    bins   = [0,25,50,75,100,150,200,600]
    labels = ["0-25","25-50","50-75","75-100","100-150","150-200","200+"]
    y_bin  = pd.cut(yte.values,bins=bins,labels=labels)
    keys   = list(MDL[suf].keys()) + ["ensemble"]
    results = {m:[np.abs(PREDS[suf][m]["test"]-yte.values)[y_bin==lb].mean()
                  if (y_bin==lb).sum()>0 else 0 for lb in labels] for m in keys}
    x=np.arange(len(labels)); bw=0.75/len(keys)
    fig,ax = plt.subplots(figsize=(13,5))
    for i,(m,col) in enumerate(zip(keys,PAL)):
        ax.bar(x+i*bw,results[m],bw,label=FNAME[m],color=col,alpha=0.85)
    ax.set_xticks(x+bw*(len(keys)-1)/2); ax.set_xticklabels(labels,rotation=20,ha="right")
    ax.set_xlabel("PM2.5 bin (ug/m3)"); ax.set_ylabel("MAE (ug/m3)")
    ax.set_title(f"MAE by PM2.5 Bin -- {TSUF[suf]}"); ax.legend(fontsize=9)
    savefig(fig, od, "plot05_mae_by_pm25_bin.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 07  Feature Importances  (from logs)
# ══════════════════════════════════════════════════════════════════════════════
def plot07(od, suf):
    print(f"\n[07] {suf}")
    imp = IMP_A if suf=="A" else IMP_B
    fig,ax = plt.subplots(figsize=(9,7))
    colors = plt.cm.RdYlGn_r(np.linspace(0.15,0.85,len(imp)))
    ax.barh(imp.index[::-1],imp.values[::-1],color=colors[::-1])
    for i,(f,v) in enumerate(zip(imp.index[::-1],imp.values[::-1])):
        ax.text(v+imp.values.max()*0.005,i,f"{v:.4f}",va="center",fontsize=8)
    ax.set_xlabel("Avg Importance (RF + XGBoost)")
    ax.set_title(f"Top-20 Feature Importances -- {TSUF[suf]}")
    savefig(fig, od, "plot07_feature_importances.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 08  Correlation Heatmap
# ══════════════════════════════════════════════════════════════════════════════
def plot08(od, suf, Xte):
    print(f"\n[08] {suf}")
    imp   = IMP_A if suf=="A" else IMP_B
    top15 = [f for f in imp.index[:15] if f in Xte.columns]
    if len(top15)<2: print("  Skipped."); return
    corr = Xte[top15].corr()
    cmap = LinearSegmentedColormap.from_list("div",["#2c7bb6","white","#d7191c"])
    fig,ax = plt.subplots(figsize=(11,9))
    sns.heatmap(corr,cmap=cmap,center=0,vmin=-1,vmax=1,annot=True,fmt=".2f",
                linewidths=0.5,ax=ax,annot_kws={"size":8})
    ax.set_title(f"Feature Correlation Heatmap (Top 15) -- {TSUF[suf]}")
    savefig(fig, od, "plot08_feature_correlation_heatmap.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 09  PM2.5 vs Key Features
# ══════════════════════════════════════════════════════════════════════════════
def plot09(od, suf, Xte, yte):
    print(f"\n[09] {suf}")
    imp  = IMP_A if suf=="A" else IMP_B
    top6 = [f for f in imp.index[:6] if f in Xte.columns]
    fig, axes = plt.subplots(2,3,figsize=(15,9))
    for ax,feat,col in zip(axes.flatten(),top6,C4*2):
        x=Xte[feat].values; y=yte.values
        mask=np.isfinite(x)&np.isfinite(y)
        ax.scatter(x[mask],y[mask],alpha=0.2,s=6,color=col,rasterized=True)
        try:
            z=np.polyfit(x[mask],y[mask],1)
            xr=np.linspace(x[mask].min(),x[mask].max(),200)
            ax.plot(xr,np.poly1d(z)(xr),"k--",lw=1.2)
        except: pass
        r=np.corrcoef(x[mask],y[mask])[0,1]
        ax.set_xlabel(feat,fontsize=9); ax.set_ylabel("PM2.5 (ug/m3)")
        ax.set_title(f"{feat}  (r={r:.2f})",fontsize=9)
    for ax in axes.flatten()[len(top6):]: ax.set_visible(False)
    fig.suptitle(f"PM2.5 vs Key Features -- {TSUF[suf]}",fontsize=13,fontweight="bold")
    savefig(fig, od, "plot09_pm25_vs_features.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 10  Summary Table  (all values from training logs)
# ══════════════════════════════════════════════════════════════════════════════
def plot10(od, suf):
    print(f"\n[10] {suf}")
    rows = []
    hdr  = ["Subset / Model","R2","MAE (ug/m3)","RMSE (ug/m3)","N"]
    me = METRICS[suf]["ensemble"]
    rows.append(["Overall (ensemble)",f"{me['r2']:.4f}",f"{me['mae']:.2f}",f"{me['rmse']:.2f}","16 735"])
    rows.append(["-- By Season --","","","",""])
    for s,(r2,n) in SEASONAL[suf].items():
        rows.append([f"  {s}",f"{r2:.4f}","--","--",f"{n:,}"])
    rows.append(["-- By AOD --","","","",""])
    for flag,(r2,n) in AOD_R2[suf].items():
        rows.append([f"  {flag}",f"{r2:.4f}","--","--",f"{n:,}"])
    rows.append(["-- Bias by Quintile --","","","",""])
    for lbl,bias,n in QUINTILE_BIAS[suf]:
        rows.append([f"  {lbl}","--",f"{bias:+.2f}","--",f"{n:,}"])
    rows.append(["-- Per Model --","","","",""])
    for m in MKEYS+["ensemble"]:
        mv=METRICS[suf][m]
        rows.append([f"  {FNAME[m]}",f"{mv['r2']:.4f}",f"{mv['mae']:.2f}",f"{mv['rmse']:.2f}","16 735"])

    fig,ax = plt.subplots(figsize=(12,max(5,len(rows)*0.42+1.8)))
    ax.axis("off")
    tbl = ax.table(cellText=rows,colLabels=hdr,loc="center",cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1.15,1.55)
    for (r,c),cell in tbl.get_celld().items():
        if r==0:
            cell.set_facecolor("#1a252f"); cell.set_text_props(color="white",fontweight="bold")
        elif r>0 and rows[r-1][1]=="" and rows[r-1][2]=="":
            cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontstyle="italic")
        elif r%2==0:
            cell.set_facecolor("#ecf0f1")
        cell.set_edgecolor("#bdc3c7")
    ax.set_title(f"Performance Summary -- {TSUF[suf]}",fontsize=13,fontweight="bold",pad=14)
    savefig(fig, od, "plot10_summary_table_ensemble.png")

# ══════════════════════════════════════════════════════════════════════════════
# PLOT 11  Metric Overview
# ══════════════════════════════════════════════════════════════════════════════
def plot11(od, suf):
    print(f"\n[11] {suf}")
    keys  = MKEYS + ["ensemble"]
    names = [FNAME[m] for m in keys]
    r2s   = [METRICS[suf][m]["r2"]   for m in keys]
    maes  = [METRICS[suf][m]["mae"]  for m in keys]
    rmses = [METRICS[suf][m]["rmse"] for m in keys]
    x = np.arange(len(keys))
    fig, axes = plt.subplots(1,3,figsize=(15,5))
    for ax,vals,title,fmt in zip(axes,[r2s,maes,rmses],
            ["R2","MAE (ug/m3)","RMSE (ug/m3)"],[".4f",".2f",".2f"]):
        bars = ax.bar(x,vals,color=PAL[:len(keys)],edgecolor="white",lw=0.6)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2, v+max(vals)*0.015,
                    format(v,fmt),ha="center",va="bottom",fontsize=9,fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(names,rotation=22,ha="right")
        ax.set_title(title)
        if title=="R2": ax.set_ylim(0,1.05)
    fig.suptitle(f"Model Performance Overview -- {TSUF[suf]}",fontsize=13,fontweight="bold")
    savefig(fig, od, "plot11_metric_overview.png")

# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════
CONFIGS = [
    ("A", OUT_NOLAG,   X_tr_A, y_tr_A, X_te_A, y_te_A),
    ("B", OUT_WITHLAG, X_tr_B, y_tr_B, X_te_B, y_te_B),
]
for suf, od, Xtr, ytr, Xte, yte in CONFIGS:
    print(f"\n{'='*60}\n  Dataset {suf}  ->  {od}\n{'='*60}")
    meta = META[suf]
    plot01(od, suf, Xtr, ytr, Xte, yte)
    plot02(od, suf, yte)
    plot03(od, suf, yte, meta)
    plot04(od, suf, yte)
    plot05(od, suf, yte)
    plot07(od, suf)
    plot08(od, suf, Xte)
    plot09(od, suf, Xte, yte)
    plot10(od, suf)
    plot11(od, suf)

print(f"\n\nAll 20 plots saved.")
print(f"  Without-lag  ->  {OUT_NOLAG}")
print(f"  With-lag     ->  {OUT_WITHLAG}")


-- Loading datasets --
  A train=(49848, 66) test=(16735, 66)
  B train=(49848, 72) test=(16735, 72)

-- Loading models --
  ok  random_forest_A_no_lags.pkl
  ok  xgboost_A_no_lags.pkl
  ok  lightgbm_A_no_lags.pkl
  ok  gradient_boosting_A_no_lags.pkl
  ok  random_forest_B_with_lags.pkl
  ok  xgboost_B_with_lags.pkl
  ok  lightgbm_B_with_lags.pkl
  ok  gradient_boosting_B_with_lags.pkl

-- Predicting --

  Dataset A  ->  C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag

[01] A
  => C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag\plot01_overfitting_diagnosis.png

[02] A
  => C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag\plot02_predicted_vs_actual.png

[03] A
  => C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag\plot03_timeseries_comparison.png

[04] A
  => C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\v2reports\withoutlag\plot04_res

In [7]:
"""
PM2.5 Pipeline — Exploratory & Results Visualizations
======================================================
Covers all key findings from the README:
  - Data distributions & quality
  - AOD missingness patterns
  - Feature correlations & physics interactions
  - Model results (hardcoded from training output)
  - Feature importance
  - Seasonal & AOD-availability breakdown
  - Prediction bias analysis

Run each section independently in Jupyter — outputs are saved as PNG.
"""

# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE = Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25")
DATA = BASE / "data" / "processed"
OUT  = BASE / "reports" / "v2reports" / "features"
OUT.mkdir(parents=True, exist_ok=True)

PATH_FINAL = DATA / "final_ml_dataset.parquet"
PATH_A     = DATA / "v1" / "dataset_A_no_lags.parquet"
PATH_B     = DATA / "v1" / "dataset_B_with_lags.parquet"

# ── Style ──────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor"  : "#fafaf8",
    "axes.facecolor"    : "#fafaf8",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#e0deda",
    "grid.linewidth"    : 0.6,
    "font.family"       : "DejaVu Sans",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 11,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "legend.frameon"    : False,
    "legend.fontsize"   : 10,
    "figure.dpi"        : 150,
})

BLUE   = "#2563eb"
TEAL   = "#0d9488"
AMBER  = "#d97706"
CORAL  = "#e5533c"
PURPLE = "#7c3aed"
GRAY   = "#6b7280"
GREEN  = "#16a34a"
PINK   = "#db2777"

def savefig(name, fig=None):
    path = OUT / f"{name}.png"
    (fig or plt).savefig(path, bbox_inches="tight", dpi=150)
    print(f"  Saved → {path.name}")
    plt.close("all")

print("Loading datasets …")
df_final = pd.read_parquet(PATH_FINAL)
df_a     = pd.read_parquet(PATH_A)
df_b     = pd.read_parquet(PATH_B)

df_final["date"] = pd.to_datetime(df_final["date"])
df_a["date"]     = pd.to_datetime(df_a["date"])
df_b["date"]     = pd.to_datetime(df_b["date"])
print(f"  final_ml_dataset : {df_final.shape}")
print(f"  dataset_A        : {df_a.shape}")
print(f"  dataset_B        : {df_b.shape}")


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 1 — PM2.5 Distribution: raw vs cleaned
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[1] PM2.5 distribution …")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("PM2.5 Distribution — Raw vs Cleaned (Dataset B)", y=1.01)

# Raw
axes[0].hist(df_final["pm25"], bins=80, color=CORAL, alpha=0.8, edgecolor="white", linewidth=0.3)
axes[0].axvline(df_final["pm25"].quantile(0.005), color=BLUE,  ls="--", lw=1.5, label=f"p0.5 = {df_final['pm25'].quantile(0.005):.1f}")
axes[0].axvline(df_final["pm25"].quantile(0.995), color=AMBER, ls="--", lw=1.5, label=f"p99.5 = {df_final['pm25'].quantile(0.995):.1f}")
axes[0].set_title("Raw (before outlier removal)")
axes[0].set_xlabel("PM2.5 (µg/m³)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Cleaned
axes[1].hist(df_b["pm25"], bins=80, color=TEAL, alpha=0.8, edgecolor="white", linewidth=0.3)
axes[1].axvline(df_b["pm25"].mean(), color=BLUE,  ls="--", lw=1.5, label=f"Mean = {df_b['pm25'].mean():.1f}")
axes[1].axvline(df_b["pm25"].median(), color=AMBER, ls="--", lw=1.5, label=f"Median = {df_b['pm25'].median():.1f}")
axes[1].set_title("Cleaned (3.6 – 319.1 µg/m³)")
axes[1].set_xlabel("PM2.5 (µg/m³)")
axes[1].legend()

fig.tight_layout()
savefig("01_pm25_distribution", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 2 — AOD Missing Rate by Season & Month
# ═══════════════════════════════════════════════════════════════════════════════
print("[2] AOD missing rate by season / month …")

df_b["month"] = pd.to_numeric(df_b["month"], errors="coerce")

monthly_miss = (
    df_b.groupby("month")["AOD_missing"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"AOD_missing": "pct_missing"})
)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
season_colors = {
    **{m: "#93c5fd" for m in [12, 1, 2]},   # winter — blue
    **{m: "#fbbf24" for m in [3, 4, 5]},    # pre-monsoon — amber
    **{m: "#4ade80" for m in [6, 7, 8, 9]}, # monsoon — green
    **{m: "#f97316" for m in [10, 11]},     # post-monsoon — orange
}

fig, ax = plt.subplots(figsize=(12, 4.5))
bars = ax.bar(
    monthly_miss["month"],
    monthly_miss["pct_missing"],
    color=[season_colors[m] for m in monthly_miss["month"]],
    edgecolor="white", linewidth=0.5, width=0.7
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
            f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=9, color="#374151")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_xlabel("Month")
ax.set_ylabel("AOD Missing (%)")
ax.set_title("Monthly AOD Missing Rate — Seasonal Patterns", pad=10)
ax.set_ylim(0, 105)

legend_patches = [
    mpatches.Patch(color="#93c5fd", label="Winter (11–25%)"),
    mpatches.Patch(color="#fbbf24", label="Pre-monsoon (17–32%)"),
    mpatches.Patch(color="#4ade80", label="Monsoon (61–93%)"),
    mpatches.Patch(color="#f97316", label="Post-monsoon (26–29%)"),
]
ax.legend(handles=legend_patches, loc="upper left", ncol=2)
fig.tight_layout()
savefig("02_aod_missing_by_month", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 3 — Station AOD Quality Distribution
# ═══════════════════════════════════════════════════════════════════════════════
print("[3] Station AOD quality …")

station_quality = (
    df_b.drop_duplicates("station_name")[["station_name", "station_aod_quality", "station_aod_missing_rate"]]
    .sort_values("station_aod_missing_rate", ascending=False)
)
quality_map   = {0: "Dead (≥95%)", 1: "Poor (≥60%)", 2: "Moderate (≥40%)", 3: "Good (<40%)"}
quality_colors= {0: CORAL, 1: AMBER, 2: BLUE, 3: TEAL}
counts = station_quality["station_aod_quality"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Station AOD Quality Encoding", y=1.01)

# Bar — count by quality
axes[0].bar(
    [quality_map[k] for k in counts.index],
    counts.values,
    color=[quality_colors[k] for k in counts.index],
    edgecolor="white", linewidth=0.5
)
for i, (q, n) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, n + 0.2, str(n), ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Number of stations")
axes[0].set_title("Station count by AOD quality tier")
axes[0].set_ylim(0, counts.max() * 1.2)

# Scatter — missing rate per station
axes[1].scatter(
    range(len(station_quality)),
    station_quality["station_aod_missing_rate"] * 100,
    c=[quality_colors[q] for q in station_quality["station_aod_quality"]],
    s=40, alpha=0.8, edgecolors="white", linewidth=0.3
)
axes[1].axhline(95, color=CORAL,  ls="--", lw=1.2, label="Dead threshold 95%")
axes[1].axhline(60, color=AMBER,  ls="--", lw=1.2, label="Poor threshold 60%")
axes[1].axhline(40, color=BLUE,   ls="--", lw=1.2, label="Moderate threshold 40%")
axes[1].set_xlabel("Station index (sorted by missing rate)")
axes[1].set_ylabel("AOD Missing Rate (%)")
axes[1].set_title("Per-station AOD missing rate")
axes[1].legend(loc="lower right")

fig.tight_layout()
savefig("03_station_aod_quality", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 4 — AOD Distribution on Clear Days
# ═══════════════════════════════════════════════════════════════════════════════
print("[4] AOD distribution (clear days) …")

aod_clear = df_b[df_b["AOD_mean"] > 0]["AOD_mean"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(aod_clear, bins=80, color=BLUE, alpha=0.8, edgecolor="white", linewidth=0.3)
ax.axvline(0.1, color=CORAL,  ls="--", lw=1.5, label="0.1 — RMS error threshold (unreliable below)")
ax.axvline(0.3, color=AMBER,  ls="--", lw=1.5, label="0.3 — uncertain flag threshold")
ax.axvline(aod_clear.mean(), color=TEAL, ls="-", lw=1.8, label=f"Mean = {aod_clear.mean():.2f}")
ax.set_xlabel("AOD_mean (clear days only)")
ax.set_ylabel("Count")
ax.set_title("AOD Distribution on Clear Days — INSAT-3DR (650 nm)")
ax.legend()
fig.tight_layout()
savefig("04_aod_distribution_clear_days", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 5 — AOD vs PM2.5 Scatter (clear days)
# ═══════════════════════════════════════════════════════════════════════════════
print("[5] AOD vs PM2.5 scatter …")

clear = df_b[df_b["AOD_mean"] > 0].sample(min(8000, (df_b["AOD_mean"] > 0).sum()), random_state=42)
season_label = {0: "Winter", 1: "Pre-monsoon", 2: "Monsoon", 3: "Post-monsoon"}
season_col   = {0: BLUE, 1: AMBER, 2: GREEN, 3: CORAL}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("AOD vs PM2.5 — Clear Sky Days Only", y=1.01)

for s, g in clear.groupby("season"):
    axes[0].scatter(g["AOD_mean"], g["pm25"], c=season_col[s], s=8, alpha=0.35,
                    label=season_label[s])
axes[0].set_xlabel("AOD_mean")
axes[0].set_ylabel("PM2.5 (µg/m³)")
axes[0].set_title("Coloured by season")
axes[0].legend(markerscale=3)

# PM_proxy (AOD/PBLH) — a better predictor
proxy_clear = clear[clear["PM_proxy"] > 0]
axes[1].scatter(proxy_clear["PM_proxy"], proxy_clear["pm25"],
                c=TEAL, s=8, alpha=0.3)
z = np.polyfit(proxy_clear["PM_proxy"].clip(0, 2), proxy_clear["pm25"], 1)
xr = np.linspace(0, 2, 200)
axes[1].plot(xr, np.polyval(z, xr), color=CORAL, lw=2, label="Linear fit")
axes[1].set_xlabel("PM_proxy = AOD_mean / (PBLH_morning_min + 1)")
axes[1].set_ylabel("PM2.5 (µg/m³)")
axes[1].set_title("Physics proxy vs PM2.5 (AOD / PBLH)")
axes[1].set_xlim(0, 2)
axes[1].legend()

fig.tight_layout()
savefig("05_aod_vs_pm25_scatter", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 6 — MERRA-2 Meteorology vs PM2.5
# ═══════════════════════════════════════════════════════════════════════════════
print("[6] MERRA-2 met vars vs PM2.5 …")

met_vars = {
    "PBLH_morning_min" : ("Morning PBLH min (m)",     BLUE),
    "SPEED_mean"       : ("Mean wind speed (m/s)",    TEAL),
    "vent_coeff_mean"  : ("Ventilation coeff (mean)", AMBER),
    "inversion_proxy_max": ("Inversion proxy (max)",  CORAL),
    "PRECTOT_sum"      : ("Daily precipitation (mm)", GREEN),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("MERRA-2 Meteorological Variables vs PM2.5", fontsize=14, y=1.01)
axes = axes.flatten()

sample = df_b.sample(min(6000, len(df_b)), random_state=42)

for i, (col, (label, color)) in enumerate(met_vars.items()):
    ax = axes[i]
    x = sample[col].clip(
        sample[col].quantile(0.01),
        sample[col].quantile(0.99)
    )
    corr = sample[col].corr(sample["pm25"])
    ax.scatter(x, sample["pm25"], c=color, s=6, alpha=0.25)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("PM2.5 (µg/m³)", fontsize=9)
    ax.set_title(f"r = {corr:.3f}", fontsize=10)

# 6th panel — trap_score
ax = axes[5]
ts = sample[sample["trap_score"] > 0]["trap_score"].clip(0, sample["trap_score"].quantile(0.98))
pm = sample[sample["trap_score"] > 0]["pm25"]
corr_ts = ts.corr(pm)
ax.scatter(ts, pm, c=PURPLE, s=6, alpha=0.25)
ax.set_xlabel("Trap score (inversion / wind / PBLH)", fontsize=9)
ax.set_ylabel("PM2.5 (µg/m³)", fontsize=9)
ax.set_title(f"r = {corr_ts:.3f}", fontsize=10)

fig.tight_layout()
savefig("06_merra2_vs_pm25", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 7 — PM2.5 Lag Correlations (Dataset B)
# ═══════════════════════════════════════════════════════════════════════════════
print("[7] PM2.5 lag correlations …")

lag_cols = ["pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_roll3", "pm25_roll7", "pm25_delta1"]
lag_corrs = {c: df_b[c].dropna().corr(df_b["pm25"][df_b[c].notna()]) for c in lag_cols}
lag_labels = {
    "pm25_lag1"   : "Yesterday (lag 1)",
    "pm25_lag2"   : "2 days ago (lag 2)",
    "pm25_lag3"   : "3 days ago (lag 3)",
    "pm25_roll3"  : "3-day rolling mean",
    "pm25_roll7"  : "7-day rolling mean",
    "pm25_delta1" : "Daily Δ (lag1 − lag2)",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("PM2.5 Lag Features vs Target PM2.5 (Dataset B)", fontsize=13, y=1.01)
axes = axes.flatten()

sample_b = df_b.sample(min(6000, len(df_b)), random_state=1)

for i, col in enumerate(lag_cols):
    ax = axes[i]
    valid = sample_b[[col, "pm25"]].dropna()
    if col == "pm25_delta1":
        x = valid[col].clip(-100, 100)
    else:
        x = valid[col].clip(0, valid[col].quantile(0.99))
    ax.scatter(x, valid["pm25"], c=PURPLE, s=6, alpha=0.3)
    ax.set_xlabel(lag_labels[col], fontsize=9)
    ax.set_ylabel("PM2.5 (µg/m³)", fontsize=9)
    ax.set_title(f"r = {lag_corrs[col]:.3f}", fontsize=10)

fig.tight_layout()
savefig("07_pm25_lag_correlations", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 8 — Monthly PM2.5 Seasonality (boxplot)
# ═══════════════════════════════════════════════════════════════════════════════
print("[8] Monthly PM2.5 seasonality …")

month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
month_data = [df_b[df_b["month"] == m]["pm25"].values for m in range(1, 13)]

season_fill = {
    **{m: "#dbeafe" for m in [12, 1, 2]},
    **{m: "#fef3c7" for m in [3, 4, 5]},
    **{m: "#dcfce7" for m in [6, 7, 8, 9]},
    **{m: "#ffedd5" for m in [10, 11]},
}

fig, ax = plt.subplots(figsize=(13, 5))
bp = ax.boxplot(
    month_data,
    patch_artist=True,
    medianprops=dict(color="#111", linewidth=1.5),
    whiskerprops=dict(linewidth=0.8),
    capprops=dict(linewidth=0.8),
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
    widths=0.55
)
for i, (patch, m) in enumerate(zip(bp["boxes"], range(1, 13))):
    patch.set_facecolor(season_fill[m])
    patch.set_edgecolor("#374151")
    patch.set_linewidth(0.8)

ax.set_xticklabels(month_labels)
ax.set_xlabel("Month")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_title("Monthly PM2.5 Distribution — All Stations (Dataset B)", pad=10)

legend_patches = [
    mpatches.Patch(color="#dbeafe", label="Winter"),
    mpatches.Patch(color="#fef3c7", label="Pre-monsoon"),
    mpatches.Patch(color="#dcfce7", label="Monsoon"),
    mpatches.Patch(color="#ffedd5", label="Post-monsoon"),
]
ax.legend(handles=legend_patches, loc="upper right")
fig.tight_layout()
savefig("08_monthly_pm25_boxplot", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 9 — Stagnant vs Non-stagnant Days
# ═══════════════════════════════════════════════════════════════════════════════
print("[9] Stagnant vs non-stagnant PM2.5 …")

stag = df_b.groupby("is_stagnant")["pm25"].describe()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Stagnant Atmosphere (low wind + shallow PBLH) vs PM2.5", y=1.01)

for s, color, label in [(0, TEAL, "Non-stagnant"), (1, CORAL, "Stagnant")]:
    axes[0].hist(df_b[df_b["is_stagnant"] == s]["pm25"], bins=60, alpha=0.65,
                 color=color, label=label, edgecolor="white", linewidth=0.3, density=True)
axes[0].set_xlabel("PM2.5 (µg/m³)")
axes[0].set_ylabel("Density")
axes[0].set_title("PM2.5 distribution by stagnation")
axes[0].legend()

categories = ["Non-stagnant", "Stagnant"]
means  = [stag.loc[0, "mean"],  stag.loc[1, "mean"]]
medians= [stag.loc[0, "50%"],   stag.loc[1, "50%"]]
x = np.arange(2)
w = 0.35
bars1 = axes[1].bar(x - w/2, means,   w, label="Mean",   color=[TEAL, CORAL], alpha=0.85)
bars2 = axes[1].bar(x + w/2, medians, w, label="Median", color=[TEAL, CORAL], alpha=0.55, hatch="///")
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel("PM2.5 (µg/m³)")
axes[1].set_title("Mean & Median PM2.5")
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
axes[1].legend()

fig.tight_layout()
savefig("09_stagnant_vs_nonstagnant", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 10 — Precipitation Washout Effect
# ═══════════════════════════════════════════════════════════════════════════════
print("[10] Precipitation washout …")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Precipitation Washout Effect on PM2.5", y=1.01)

# Rainy vs dry day
for is_rain, color, label in [(0, AMBER, "Dry day"), (1, BLUE, "Rainy day")]:
    axes[0].hist(df_b[df_b["is_rainy"] == is_rain]["pm25"], bins=60, alpha=0.65,
                 color=color, label=label, edgecolor="white", linewidth=0.3, density=True)
axes[0].set_xlabel("PM2.5 (µg/m³)")
axes[0].set_ylabel("Density")
axes[0].set_title("Rainy vs dry day")
axes[0].legend()

# PM2.5 vs rain_3day
rain_bins = pd.cut(df_b["rain_3day"], bins=[0, 0.5, 2, 5, 10, 50, 200],
                   labels=["0", "0.5–2", "2–5", "5–10", "10–50", ">50"])
rain_pm = df_b.groupby(rain_bins, observed=True)["pm25"].median()
axes[1].bar(range(len(rain_pm)), rain_pm.values, color=BLUE, alpha=0.8, edgecolor="white")
axes[1].set_xticks(range(len(rain_pm)))
axes[1].set_xticklabels(rain_pm.index, fontsize=9)
axes[1].set_xlabel("3-day cumulative precipitation (mm)")
axes[1].set_ylabel("Median PM2.5 (µg/m³)")
axes[1].set_title("Wet scavenging by 3-day rainfall")

# Post-rain effect
for lag, color, label in [("post_rain1", TEAL, "1 day after rain"), ("post_rain2", GREEN, "2 days after rain")]:
    for flag, ls in [(0, "-"), (1, "--")]:
        vals = df_b[df_b[lag] == flag]["pm25"]
        axes[2].hist(vals, bins=50, alpha=0.55, color=color, density=True,
                     ls=ls, edgecolor="white", linewidth=0.2,
                     label=f"{label} ({'yes' if flag else 'no'})")
axes[2].set_xlabel("PM2.5 (µg/m³)")
axes[2].set_ylabel("Density")
axes[2].set_title("Post-rain PM2.5 suppression")
axes[2].legend(fontsize=8)

fig.tight_layout()
savefig("10_precipitation_washout", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 11 — R² Progression Through Pipeline (README Table)
# ═══════════════════════════════════════════════════════════════════════════════
print("[11] R² pipeline progression …")

stages = [
    "Naive AOD only",
    "+ MERRA meteorology",
    "+ Physics feature engineering",
    "+ PM2.5 lag features ★",
    "+ AOD lags + rolling",
    "+ Station spatial context",
    "+ Ensemble (final)",
]
r2_vals = [0.475, 0.625, 0.710, 0.835, 0.840, 0.848, 0.852]
colors  = [GRAY, BLUE, TEAL, CORAL, AMBER, PURPLE, GREEN]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(stages, r2_vals, color=colors, edgecolor="white", linewidth=0.5, height=0.55)

for bar, val in zip(bars, r2_vals):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10, fontweight="bold", color="#111")

ax.set_xlim(0, 0.93)
ax.set_xlabel("R² (Ensemble — Dataset B)")
ax.set_title("R² Progression Through the Pipeline", pad=10)
ax.invert_yaxis()

# Highlight the biggest jump
ax.barh(stages[3], r2_vals[3], color=CORAL, edgecolor=CORAL, linewidth=1.5,
        height=0.55, alpha=0.15)
ax.annotate("Largest single gain\n+0.13 R²", xy=(0.835, 3), xytext=(0.68, 3),
            arrowprops=dict(arrowstyle="->", color=CORAL, lw=1.5),
            fontsize=9, color=CORAL, va="center")

fig.tight_layout()
savefig("11_r2_pipeline_progression", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 12 — Dataset A vs B Model Comparison (README Tables)
# ═══════════════════════════════════════════════════════════════════════════════
print("[12] Dataset A vs B model comparison …")

models = ["Random Forest", "XGBoost", "LightGBM", "Gradient Boosting", "Ensemble"]
r2_a   = [0.7073, 0.7214, 0.7200, 0.7023, 0.7211]
r2_b   = [0.8483, 0.8514, 0.8507, 0.8480, 0.8522]
mae_a  = [15.67,  15.25,  15.28,  15.65,  15.25 ]
mae_b  = [10.40,  10.37,  10.44,  10.43,  10.32 ]
rmse_a = [24.59,  23.99,  24.05,  24.80,  24.00 ]
rmse_b = [17.70,  17.52,  17.56,  17.72,  17.47 ]

x = np.arange(len(models))
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Dataset A (no lags) vs Dataset B (with PM2.5 lags) — Model Comparison", fontsize=13)

for ax, metric, a_vals, b_vals, ylabel, title in zip(
    axes,
    ["R²", "MAE", "RMSE"],
    [r2_a, mae_a, rmse_a],
    [r2_b, mae_b, rmse_b],
    ["R²", "MAE (µg/m³)", "RMSE (µg/m³)"],
    ["R² (higher = better)", "MAE (lower = better)", "RMSE (lower = better)"]
):
    b1 = ax.bar(x - w/2, a_vals, w, label="Dataset A — no lags", color=BLUE,   alpha=0.75, edgecolor="white")
    b2 = ax.bar(x + w/2, b_vals, w, label="Dataset B — with lags", color=CORAL, alpha=0.85, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

    # Annotate delta
    for xi, (va, vb) in enumerate(zip(a_vals, b_vals)):
        delta = vb - va
        sign  = "+" if delta > 0 else ""
        ax.text(xi, max(va, vb) + (0.005 if metric == "R²" else 0.3),
                f"{sign}{delta:.2f}", ha="center", fontsize=7.5,
                color=GREEN if (metric == "R²" and delta > 0) else CORAL)

fig.tight_layout()
savefig("12_dataset_A_vs_B_comparison", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 13 — R² Gain per Model from PM2.5 Lags
# ═══════════════════════════════════════════════════════════════════════════════
print("[13] R² gain from lags …")

gains = [b - a for a, b in zip(r2_a, r2_b)]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(models, gains, color=[CORAL if i < 4 else AMBER for i in range(5)],
              edgecolor="white", linewidth=0.5, width=0.5)
for bar, g in zip(bars, gains):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"+{g:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold", color="#111")
ax.set_ylabel("R² gain (B − A)")
ax.set_title("R² Gain from Adding PM2.5 Lag Features", pad=10)
ax.set_ylim(0, 0.17)
fig.tight_layout()
savefig("13_r2_gain_from_lags", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 14 — Seasonal R² Breakdown (README Table)
# ═══════════════════════════════════════════════════════════════════════════════
print("[14] Seasonal R² …")

seasons     = ["Winter\n(Dec–Feb)", "Monsoon\n(Jun–Sep)", "Post-monsoon\n(Oct–Nov)"]
season_r2   = [0.8156, 0.6403, 0.8412]
season_rows = [2828,   5593,   8314  ]
season_clr  = [BLUE, GREEN, AMBER]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Seasonal R² — Dataset B Ensemble", y=1.01)

bars = axes[0].bar(seasons, season_r2, color=season_clr, edgecolor="white", linewidth=0.5, width=0.5)
for bar, val in zip(bars, season_r2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].axhline(0.8522, color=GRAY, ls="--", lw=1.2, label="Overall ensemble R² = 0.8522")
axes[0].set_ylim(0, 0.95)
axes[0].set_ylabel("R²")
axes[0].set_title("R² by season")
axes[0].legend()

# Explain monsoon weakness
aod_by_season = {
    0: ("Winter",       "#93c5fd", 18),
    1: ("Pre-monsoon",  "#fbbf24", 25),
    2: ("Monsoon",      "#4ade80", 77),
    3: ("Post-monsoon", "#f97316", 27),
}
s_labels = [v[0] for v in aod_by_season.values()]
s_miss   = [v[2] for v in aod_by_season.values()]
s_colors = [v[1] for v in aod_by_season.values()]
axes[1].bar(s_labels, s_miss, color=s_colors, edgecolor="white", linewidth=0.5, width=0.5)
for i, m in enumerate(s_miss):
    axes[1].text(i, m + 0.5, f"{m}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Avg AOD Missing (%)")
axes[1].set_title("AOD missing rate by season\n(explains low monsoon R²)")
axes[1].set_ylim(0, 100)

fig.tight_layout()
savefig("14_seasonal_r2_breakdown", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 15 — R² by AOD Availability
# ═══════════════════════════════════════════════════════════════════════════════
print("[15] R² by AOD availability …")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Performance on Cloud Days (AOD Missing) vs Clear Days", y=1.01)

categories = ["Cloud days\n(AOD = −1)", "Clear days\n(AOD > 0)"]
r2_vals_aod = [0.7310, 0.8335]
rows_aod    = [9185, 7550]
colors_aod  = [BLUE, TEAL]

bars = axes[0].bar(categories, r2_vals_aod, color=colors_aod, edgecolor="white", linewidth=0.5, width=0.45)
for bar, val in zip(bars, r2_vals_aod):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.4f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[0].set_ylim(0, 0.95)
axes[0].set_ylabel("R²")
axes[0].set_title("R² by AOD retrieval status")
axes[0].annotate("Gap = 0.10\n(PM2.5 lags carry\ncloud-day signal)",
                 xy=(1, 0.7310 + 0.03), xytext=(0.5, 0.76),
                 arrowprops=dict(arrowstyle="->", color=GRAY, lw=1),
                 ha="center", fontsize=9, color=GRAY)

# Donut — composition of test set
sizes  = [9185, 7550]
labels = [f"Cloud days\n(AOD = −1)\n43.4% — R²=0.731",
          f"Clear days\n(AOD > 0)\n56.6% — R²=0.834"]
wedge_colors = [BLUE, TEAL]
wedges, texts = axes[1].pie(sizes, labels=None, colors=wedge_colors,
                             startangle=90, wedgeprops=dict(width=0.5, edgecolor="white"))
axes[1].legend(wedges, labels, loc="center left", bbox_to_anchor=(0.75, 0.5), fontsize=9)
axes[1].set_title("Test set composition")

fig.tight_layout()
savefig("15_r2_by_aod_availability", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 16 — Prediction Bias by PM2.5 Quintile (README Table)
# ═══════════════════════════════════════════════════════════════════════════════
print("[16] Prediction bias by quintile …")

quintiles = ["Q1 — Low\n(< 20 µg/m³)", "Q2", "Q3", "Q4", "Q5 — High\n(> 100 µg/m³)"]
biases    = [+4.36, +2.84, +1.11, -0.53, -7.61]
colors_q  = [CORAL if b > 0 else BLUE for b in biases]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(quintiles, biases, color=colors_q, edgecolor="white", linewidth=0.5, width=0.55)
ax.axhline(0, color="#111", lw=0.8, ls="-")
for bar, val in zip(bars, biases):
    sign = "+" if val > 0 else ""
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.15 if val > 0 else -0.4),
            f"{sign}{val:.2f}", ha="center", va="bottom" if val > 0 else "top",
            fontsize=11, fontweight="bold", color="#111")
ax.set_ylabel("Bias (Predicted − Actual) µg/m³")
ax.set_title("Prediction Bias by PM2.5 Quintile — Regression to the Mean Effect", pad=10)
ax.set_ylim(-10, 6.5)

legend_patches = [
    mpatches.Patch(color=CORAL, label="Overprediction (bias > 0)"),
    mpatches.Patch(color=BLUE,  label="Underprediction (bias < 0)"),
]
ax.legend(handles=legend_patches)
fig.tight_layout()
savefig("16_prediction_bias_by_quintile", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 17 — Top 20 Feature Importances (README Table)
# ═══════════════════════════════════════════════════════════════════════════════
print("[17] Top 20 feature importances …")

feat_importance = {
    "pm25_lag1"          : 0.4623,
    "pm25_roll3"         : 0.2156,
    "pm25_roll7"         : 0.0623,
    "station_month_pm"   : 0.0570,
    "rain_3day"          : 0.0089,
    "pm25_lag2"          : 0.0083,
    "rain_lag1"          : 0.0083,
    "SPEED_mean"         : 0.0080,
    "vent_coeff_mean"    : 0.0072,
    "pm25_delta1"        : 0.0063,
    "AOD_x_vent"         : 0.0058,
    "SPEED_min"          : 0.0051,
    "pm25_lag3"          : 0.0049,
    "doy_sin"            : 0.0048,
    "PRECTOT_sum"        : 0.0047,
    "month_cos"          : 0.0047,
    "moisture_max"       : 0.0044,
    "vent_coeff_min"     : 0.0043,
    "moisture_mean"      : 0.0041,
    "doy_cos"            : 0.0041,
}

groups = {
    "PM2.5 lags / rolling" : CORAL,
    "Station context"      : PURPLE,
    "Precipitation"        : BLUE,
    "MERRA-2 met"          : TEAL,
    "Temporal cyclical"    : AMBER,
    "AOD interaction"      : GREEN,
}
feat_group = {
    "pm25_lag1"        : "PM2.5 lags / rolling",
    "pm25_roll3"       : "PM2.5 lags / rolling",
    "pm25_roll7"       : "PM2.5 lags / rolling",
    "pm25_lag2"        : "PM2.5 lags / rolling",
    "pm25_delta1"      : "PM2.5 lags / rolling",
    "pm25_lag3"        : "PM2.5 lags / rolling",
    "station_month_pm" : "Station context",
    "rain_3day"        : "Precipitation",
    "rain_lag1"        : "Precipitation",
    "PRECTOT_sum"      : "Precipitation",
    "SPEED_mean"       : "MERRA-2 met",
    "vent_coeff_mean"  : "MERRA-2 met",
    "SPEED_min"        : "MERRA-2 met",
    "moisture_max"     : "MERRA-2 met",
    "vent_coeff_min"   : "MERRA-2 met",
    "moisture_mean"    : "MERRA-2 met",
    "doy_sin"          : "Temporal cyclical",
    "doy_cos"          : "Temporal cyclical",
    "month_cos"        : "Temporal cyclical",
    "AOD_x_vent"       : "AOD interaction",
}

feat_df = pd.DataFrame({
    "feature"    : list(feat_importance.keys()),
    "importance" : list(feat_importance.values()),
    "group"      : [feat_group[f] for f in feat_importance.keys()]
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
colors_bar = [groups[g] for g in feat_df["group"]]
bars = ax.barh(feat_df["feature"], feat_df["importance"], color=colors_bar,
               edgecolor="white", linewidth=0.4, height=0.65)
for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.4f}", va="center", fontsize=8.5)
ax.set_xlabel("Feature Importance (avg RF + XGBoost)")
ax.set_title("Top 20 Feature Importances — Dataset B", pad=10)
ax.set_xlim(0, 0.52)

legend_patches = [mpatches.Patch(color=c, label=g) for g, c in groups.items()]
ax.legend(handles=legend_patches, loc="lower right", fontsize=9)
fig.tight_layout()
savefig("17_top20_feature_importance", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 18 — Correlation Heatmap (Key Features vs PM2.5)
# ═══════════════════════════════════════════════════════════════════════════════
print("[18] Correlation heatmap …")

key_cols = [
    "pm25", "pm25_lag1", "pm25_roll3", "pm25_roll7",
    "AOD_mean", "PM_proxy", "trap_score",
    "PBLH_morning_min", "SPEED_mean", "vent_coeff_mean",
    "inversion_proxy_max", "moisture_mean", "PRECTOT_sum",
    "station_pm_mean", "station_month_pm"
]
corr_df = df_b[key_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
im = ax.imshow(corr_df.values, cmap="RdBu_r", norm=norm, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")

ax.set_xticks(range(len(key_cols)))
ax.set_yticks(range(len(key_cols)))
ax.set_xticklabels(key_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(key_cols, fontsize=9)

for i in range(len(key_cols)):
    for j in range(len(key_cols)):
        val = corr_df.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(val) > 0.5 else "#111")

ax.set_title("Correlation Matrix — Key Features vs PM2.5", pad=12, fontsize=13)
fig.tight_layout()
savefig("18_correlation_heatmap", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 19 — Station-level PM2.5 Mean (Geographic Gradient — top/bottom stations)
# ═══════════════════════════════════════════════════════════════════════════════
print("[19] Station PM2.5 means …")

station_pm = (
    df_b.groupby("station_name")["pm25"]
    .agg(["mean", "std", "median"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
n = len(station_pm)
colors_station = [CORAL if i < n//3 else (AMBER if i < 2*n//3 else TEAL)
                  for i in range(n)]
ax.bar(range(n), station_pm["mean"], color=colors_station,
       edgecolor="white", linewidth=0.2, width=1.0)
ax.fill_between(range(n),
                station_pm["mean"] - station_pm["std"],
                station_pm["mean"] + station_pm["std"],
                alpha=0.15, color=GRAY)
ax.set_xlabel("Station (ranked by mean PM2.5)")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_title("Station-level Mean PM2.5 — IGP High-Pollution vs Southern Cleaner Stations", pad=10)
ax.set_xticks([])

legend_patches = [
    mpatches.Patch(color=CORAL, label="Top third — high pollution (IGP)"),
    mpatches.Patch(color=AMBER, label="Middle third"),
    mpatches.Patch(color=TEAL,  label="Bottom third — cleaner (south/coastal)"),
    mpatches.Patch(color=GRAY,  alpha=0.3, label="±1 SD band"),
]
ax.legend(handles=legend_patches, loc="upper right")
fig.tight_layout()
savefig("19_station_pm25_ranking", fig)


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 20 — Summary Dashboard (one-page overview)
# ═══════════════════════════════════════════════════════════════════════════════
print("[20] Summary dashboard …")

fig = plt.figure(figsize=(16, 10))
fig.suptitle("PM2.5 Estimation Pipeline — Summary Dashboard", fontsize=15, fontweight="bold", y=1.01)
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.35)

# (A) R² Progression
ax_a = fig.add_subplot(gs[0, 0])
short_stages = ["AOD only", "+MERRA", "+Physics", "+PM lags ★", "+AOD lags", "+Station", "Ensemble"]
ax_a.barh(short_stages, r2_vals, color=colors, edgecolor="white", linewidth=0.4, height=0.6)
ax_a.set_xlim(0, 0.93)
ax_a.set_xlabel("R²")
ax_a.set_title("R² Pipeline Progression")
ax_a.invert_yaxis()
for i, (v, s) in enumerate(zip(r2_vals, short_stages)):
    ax_a.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)

# (B) A vs B R² comparison
ax_b = fig.add_subplot(gs[0, 1])
x = np.arange(len(models))
ax_b.bar(x - 0.2, r2_a, 0.35, label="Dataset A", color=BLUE,  alpha=0.75, edgecolor="white")
ax_b.bar(x + 0.2, r2_b, 0.35, label="Dataset B", color=CORAL, alpha=0.85, edgecolor="white")
ax_b.set_xticks(x)
ax_b.set_xticklabels(["RF","XGB","LGBM","GB","Ens."], fontsize=9)
ax_b.set_ylabel("R²")
ax_b.set_title("Dataset A vs B — R²")
ax_b.legend(fontsize=8)
ax_b.set_ylim(0.65, 0.88)

# (C) Top 10 Feature Importance
ax_c = fig.add_subplot(gs[0, 2])
top10 = list(feat_importance.items())[:10]
f_names = [x[0] for x in top10]
f_vals  = [x[1] for x in top10]
f_colors= [groups[feat_group[f]] for f in f_names]
ax_c.barh(f_names[::-1], f_vals[::-1], color=f_colors[::-1], edgecolor="white", linewidth=0.4, height=0.65)
ax_c.set_xlabel("Importance")
ax_c.set_title("Top 10 Feature Importances")

# (D) Monthly PM2.5
ax_d = fig.add_subplot(gs[1, 0])
monthly_pm = df_b.groupby("month")["pm25"].median()
ax_d.plot(monthly_pm.index, monthly_pm.values, marker="o", color=BLUE, lw=2, ms=5)
ax_d.fill_between(monthly_pm.index, monthly_pm.values, alpha=0.1, color=BLUE)
ax_d.set_xticks(range(1, 13))
ax_d.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax_d.set_ylabel("Median PM2.5 (µg/m³)")
ax_d.set_title("Monthly PM2.5 Seasonality")

# (E) Seasonal R²
ax_e = fig.add_subplot(gs[1, 1])
ax_e.bar(["Winter", "Monsoon", "Post-\nmonsoon"],
         [0.8156, 0.6403, 0.8412],
         color=[BLUE, GREEN, AMBER], edgecolor="white", linewidth=0.5, width=0.45)
ax_e.axhline(0.8522, color=GRAY, ls="--", lw=1.2, label="Overall 0.8522")
ax_e.set_ylim(0, 0.95)
ax_e.set_ylabel("R²")
ax_e.set_title("Seasonal R² (Dataset B Ensemble)")
ax_e.legend(fontsize=8)

# (F) Prediction bias
ax_f = fig.add_subplot(gs[1, 2])
q_labels = ["Q1\nLow","Q2","Q3","Q4","Q5\nHigh"]
q_cols   = [CORAL if b > 0 else BLUE for b in biases]
ax_f.bar(q_labels, biases, color=q_cols, edgecolor="white", linewidth=0.5, width=0.55)
ax_f.axhline(0, color="#111", lw=0.8)
ax_f.set_ylabel("Bias (µg/m³)")
ax_f.set_title("Prediction Bias by PM2.5 Quintile")
for i, b in enumerate(biases):
    sign = "+" if b > 0 else ""
    ax_f.text(i, b + (0.2 if b > 0 else -0.5),
              f"{sign}{b:.2f}", ha="center", va="bottom" if b > 0 else "top", fontsize=8)

fig.tight_layout()
savefig("20_summary_dashboard", fig)


# ─────────────────────────────────────────────────────────────────────────────
print(f"\n✓ All 20 plots saved to:\n  {OUT}")

Loading datasets …
  final_ml_dataset : (67970, 28)
  dataset_A        : (66583, 73)
  dataset_B        : (66583, 79)

[1] PM2.5 distribution …
  Saved → 01_pm25_distribution.png
[2] AOD missing rate by season / month …
  Saved → 02_aod_missing_by_month.png
[3] Station AOD quality …
  Saved → 03_station_aod_quality.png
[4] AOD distribution (clear days) …
  Saved → 04_aod_distribution_clear_days.png
[5] AOD vs PM2.5 scatter …
  Saved → 05_aod_vs_pm25_scatter.png
[6] MERRA-2 met vars vs PM2.5 …
  Saved → 06_merra2_vs_pm25.png
[7] PM2.5 lag correlations …
  Saved → 07_pm25_lag_correlations.png
[8] Monthly PM2.5 seasonality …
  Saved → 08_monthly_pm25_boxplot.png
[9] Stagnant vs non-stagnant PM2.5 …
  Saved → 09_stagnant_vs_nonstagnant.png
[10] Precipitation washout …
  Saved → 10_precipitation_washout.png
[11] R² pipeline progression …
  Saved → 11_r2_pipeline_progression.png
[12] Dataset A vs B model comparison …
  Saved → 12_dataset_A_vs_B_comparison.png
[13] R² gain from lags …
  Saved